In [ ]:

# ============================================================
# EXPERIMENT 19 - THREE NEW FPN ARCHITECTURES
# ============================================================

import os
import json
import time
import random
import shutil
import warnings

from pathlib import Path, PurePosixPath
from concurrent.futures import ThreadPoolExecutor

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA GPU unavailable."

DEVICE = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))

print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

USE_AMP = True

IMAGE_SIZE = 224

NUM_WORKERS = 2

print("\nEnvironment ready.")


In [ ]:

# ============================================================
# LOAD CORRECTED MANIFEST
# ============================================================

cwd = Path.cwd().resolve()

PROJECT_ROOT = (
    cwd.parent
    if cwd.name.lower() == "notebooks"
    else cwd
)

manifest_candidates = [

    PROJECT_ROOT /
    "Notebooks/reports/eda/exp16_duplicate_detection/"
    "dataset_split_manifest_corrected.csv",

    PROJECT_ROOT /
    "reports/eda/exp16_duplicate_detection/"
    "dataset_split_manifest_corrected.csv"

]

MANIFEST_PATH = next(
    (
        p for p in manifest_candidates
        if p.exists()
    ),
    None
)

assert MANIFEST_PATH is not None, (
    "Corrected manifest not found. Check its location."
)

split_df = pd.read_csv(MANIFEST_PATH)

required = [
    "s3_key",
    "folder_1",
    "folder_2",
    "split"
]

assert set(required).issubset(split_df.columns)

assert len(split_df) == 14565

assert split_df["s3_key"].is_unique

assert split_df[required].notna().all().all()

split_df["class_id"] = (
    split_df["folder_2"]
    .str.extract(r"^(\d+)")
    .astype(int)
)

NUM_CLASSES = 11

assert set(split_df["class_id"].unique()) == set(range(11))

class_names = (
    split_df[
        ["class_id", "folder_2"]
    ]
    .drop_duplicates()
    .sort_values("class_id")
    .set_index("class_id")["folder_2"]
    .to_dict()
)

assert len(class_names) == NUM_CLASSES

train_df = (
    split_df[split_df["split"] == "train"]
    .reset_index(drop=True)
)

val_df = (
    split_df[split_df["split"] == "validation"]
    .reset_index(drop=True)
)

test_df = (
    split_df[split_df["split"] == "test"]
    .reset_index(drop=True)
)

assert len(train_df) + len(val_df) + len(test_df) == 14565

print("DATASET RECOVERED")
print("=================")

print("Total:", len(split_df))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Fabrics:", split_df["folder_1"].nunique())
print("Classes:", NUM_CLASSES)

display(
    pd.crosstab(
        split_df["folder_2"],
        split_df["split"]
    )
)

# ------------------------------------------------------------
# Class-balanced loss weights
# ------------------------------------------------------------

class_counts = (
    train_df["class_id"]
    .value_counts()
    .reindex(range(NUM_CLASSES), fill_value=0)
    .values
)

class_weights_np = np.sqrt(
    class_counts.sum()
    / (NUM_CLASSES * class_counts)
)

class_weights_np /= class_weights_np.mean()

CLASS_WEIGHTS = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

print("\nManifest and class weights ready.")


In [ ]:

# ============================================================
# S3 AND LOCAL ORIGINAL IMAGE CACHE
# ============================================================

BUCKET_NAME = (
    "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
)

REGION = "ap-southeast-2"

s3 = boto3.client(
    "s3",
    region_name=REGION
)

CACHE_ROOT = (
    PROJECT_ROOT /
    "data" /
    "training_cache" /
    "original"
)

CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("S3 connected.")

disk = shutil.disk_usage(CACHE_ROOT)

print(
    "Available disk:",
    round(disk.free / 1024**3, 2),
    "GB"
)

keys = split_df["s3_key"].unique().tolist()

def local_image_path(key):

    return CACHE_ROOT.joinpath(
        *PurePosixPath(key).parts
    )

download_tasks = []

for key in keys:

    path = local_image_path(key)

    if not path.exists() or path.stat().st_size == 0:

        download_tasks.append((key, path))


print("Expected images:", len(keys))
print("Already cached:", len(keys) - len(download_tasks))
print("To download:", len(download_tasks))


def download_one(task):

    key, path = task

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    s3.download_file(
        BUCKET_NAME,
        key,
        str(path)
    )

    return key


if download_tasks:

    with ThreadPoolExecutor(max_workers=8) as executor:

        list(
            tqdm(
                executor.map(
                    download_one,
                    download_tasks
                ),
                total=len(download_tasks),
                desc="Downloading original images"
            )
        )


missing = [

    key for key in keys

    if (
        not local_image_path(key).exists()
        or local_image_path(key).stat().st_size == 0
    )

]

assert len(missing) == 0, (
    f"Missing images: {len(missing)}"
)

print("\nORIGINAL IMAGE CACHE VERIFIED")

print(
    "Cached storage (GB):",
    round(
        sum(
            local_image_path(k).stat().st_size
            for k in keys
        ) / 1024**3,
        3
    )
)


In [ ]:

# ============================================================
# DATASET AND DATALOADERS
# ============================================================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomAffine(
        degrees=4,
        translate=(0.02, 0.02),
        scale=(0.97, 1.03),
        fill=(128, 128, 128)
    ),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.08,
                contrast=0.08,
                saturation=0.05,
                hue=0.01
            )
        ],
        p=0.5
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )

])

evaluation_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )

])


class StitchingDataset(Dataset):

    def __init__(
        self,
        dataframe,
        training=False
    ):

        self.df = dataframe.reset_index(drop=True)

        self.training = training

        self.transform = (
            train_transform
            if training
            else evaluation_transform
        )

        self.paths = [

            local_image_path(key)

            for key in self.df["s3_key"]

        ]

        self.labels = (
            self.df["class_id"]
            .astype(int)
            .tolist()
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        with Image.open(self.paths[index]) as img:

            image = ImageOps.exif_transpose(
                img
            ).convert("RGB")

        image = self.transform(image)

        return (
            image,
            self.labels[index],
            index
        )


def make_loaders(batch_size):

    train_dataset = StitchingDataset(
        train_df,
        training=True
    )

    val_dataset = StitchingDataset(
        val_df,
        training=False
    )

    test_dataset = StitchingDataset(
        test_df,
        training=False
    )

    generator = torch.Generator()

    generator.manual_seed(SEED)

    common = {

        "batch_size": batch_size,

        "num_workers": NUM_WORKERS,

        "pin_memory": True,

        "persistent_workers": True

    }

    train_loader = DataLoader(

        train_dataset,

        shuffle=True,

        generator=generator,

        **common

    )

    val_loader = DataLoader(

        val_dataset,

        shuffle=False,

        **common

    )

    test_loader = DataLoader(

        test_dataset,

        shuffle=False,

        **common

    )

    return train_loader, val_loader, test_loader


def make_eval_loader(dataframe, batch_size):

    dataset = StitchingDataset(
        dataframe,
        training=False
    )

    return DataLoader(

        dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=True,

        persistent_workers=True

    )


print("Dataset and DataLoader functions ready.")


In [ ]:

# ============================================================
# GENERALIZED FPN CLASSIFICATION HEAD
# ============================================================

class GeM(nn.Module):

    def __init__(self, p=3.0, eps=1e-6):

        super().__init__()

        self.p = nn.Parameter(
            torch.ones(1) * p
        )

        self.eps = eps

    def forward(self, x):

        p = torch.clamp(
            self.p,
            min=1.0,
            max=6.0
        )

        x = x.clamp(
            min=self.eps
        ).pow(p)

        x = F.adaptive_avg_pool2d(
            x,
            1
        )

        return x.pow(
            1.0 / p
        ).flatten(1)


class FPNHead(nn.Module):

    def __init__(
        self,
        input_channels,
        num_classes=11,
        fpn_channels=256,
        dropout=0.3
    ):

        super().__init__()

        def lateral_block(in_channels):

            return nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    fpn_channels,
                    kernel_size=1,
                    bias=False
                ),

                nn.GroupNorm(
                    32,
                    fpn_channels
                ),

                nn.ReLU(inplace=True)

            )

        def smooth_block():

            return nn.Sequential(

                nn.Conv2d(
                    fpn_channels,
                    fpn_channels,
                    kernel_size=3,
                    padding=1,
                    bias=False
                ),

                nn.GroupNorm(
                    32,
                    fpn_channels
                ),

                nn.ReLU(inplace=True)

            )

        self.lateral = nn.ModuleList([

            lateral_block(c)
            for c in input_channels

        ])

        self.smooth = nn.ModuleList([

            smooth_block()
            for _ in range(4)

        ])

        self.dropout2d = nn.Dropout2d(0.10)

        self.gem = GeM()

        self.classifier = nn.Sequential(

            nn.Linear(
                fpn_channels * 4,
                512
            ),

            nn.LayerNorm(512),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                512,
                num_classes
            )

        )

    def forward(self, c2, c3, c4, c5):

        features = [c2, c3, c4, c5]

        l2, l3, l4, l5 = [

            self.lateral[i](features[i])

            for i in range(4)

        ]

        # Top-down feature fusion

        p5 = l5

        p4 = l4 + F.interpolate(
            p5,
            size=l4.shape[-2:],
            mode="nearest"
        )

        p3 = l3 + F.interpolate(
            p4,
            size=l3.shape[-2:],
            mode="nearest"
        )

        p2 = l2 + F.interpolate(
            p3,
            size=l2.shape[-2:],
            mode="nearest"
        )

        # Refine each feature level

        pyramid = [

            self.smooth[0](p2),

            self.smooth[1](p3),

            self.smooth[2](p4),

            self.smooth[3](p5)

        ]

        # Multi-scale GeM pooling

        pooled = [

            self.gem(
                self.dropout2d(feature)
            )

            for feature in pyramid

        ]

        combined = torch.cat(
            pooled,
            dim=1
        )

        return self.classifier(combined)


# ============================================================
# BACKBONE + FPN CLASSIFIER
# ============================================================

class FPNClassifier(nn.Module):

    def __init__(
        self,
        architecture,
        num_classes=11,
        pretrained=True,
        dropout=0.3
    ):

        super().__init__()

        self.architecture = architecture

        if architecture == "densenet121_fpn":

            weights = (
                models.DenseNet121_Weights.DEFAULT
                if pretrained else None
            )

            backbone = models.densenet121(
                weights=weights
            )

            backbone.classifier = nn.Identity()

            channels = [
                256, 512, 1024, 1024
            ]

            self.backbone_type = "densenet121"

        elif architecture == "convnext_tiny_fpn":

            weights = (
                models.ConvNeXt_Tiny_Weights.DEFAULT
                if pretrained else None
            )

            backbone = models.convnext_tiny(
                weights=weights
            )

            backbone.classifier = nn.Identity()

            channels = [
                96, 192, 384, 768
            ]

            self.backbone_type = "convnext_tiny"

        elif architecture == "efficientnet_b0_fpn":

            weights = (
                models.EfficientNet_B0_Weights.DEFAULT
                if pretrained else None
            )

            backbone = models.efficientnet_b0(
                weights=weights
            )

            backbone.classifier = nn.Identity()

            channels = [
                24, 40, 112, 1280
            ]

            self.backbone_type = "efficientnet_b0"

        else:

            raise ValueError(
                f"Unknown architecture: {architecture}"
            )

        self.backbone = backbone

        self.fpn_head = FPNHead(

            input_channels=channels,

            num_classes=num_classes,

            fpn_channels=256,

            dropout=dropout

        )

    def extract_features(self, x):

        f = self.backbone.features

        # ----------------------------------------------------
        # DenseNet121
        # ----------------------------------------------------

        if self.backbone_type == "densenet121":

            x = f.conv0(x)
            x = f.norm0(x)
            x = f.relu0(x)
            x = f.pool0(x)

            c2 = f.denseblock1(x)

            x = f.transition1(c2)

            c3 = f.denseblock2(x)

            x = f.transition2(c3)

            c4 = f.denseblock3(x)

            x = f.transition3(c4)

            c5 = f.denseblock4(x)

            c5 = F.relu(
                f.norm5(c5),
                inplace=False
            )

        # ----------------------------------------------------
        # ConvNeXt-Tiny
        # ----------------------------------------------------

        elif self.backbone_type == "convnext_tiny":

            x = f[0](x)

            c2 = f[1](x)

            x = f[2](c2)

            c3 = f[3](x)

            x = f[4](c3)

            c4 = f[5](x)

            x = f[6](c4)

            c5 = f[7](x)

        # ----------------------------------------------------
        # EfficientNet-B0
        # ----------------------------------------------------

        elif self.backbone_type == "efficientnet_b0":

            x = f[0](x)

            x = f[1](x)

            c2 = f[2](x)

            c3 = f[3](c2)

            x = f[4](c3)

            c4 = f[5](x)

            x = f[6](c4)

            x = f[7](x)

            c5 = f[8](x)

        return c2, c3, c4, c5

    def forward(self, x):

        features = self.extract_features(x)

        return self.fpn_head(*features)


def create_fpn_model(
    architecture,
    pretrained=True,
    dropout=0.3
):

    return FPNClassifier(

        architecture=architecture,

        num_classes=NUM_CLASSES,

        pretrained=pretrained,

        dropout=dropout

    )


print("Three FPN architectures implemented.")


In [ ]:

# ============================================================
# ARCHITECTURE-SPECIFIC CONFIGURATIONS
# ============================================================

MODEL_CONFIGS = {

    "densenet121_fpn": {

        "batch_size": 16,
        "accumulation_steps": 1,

        "max_epochs": 60,
        "early_stopping_patience": 10,

        "backbone_lr": 3e-5,
        "head_lr": 3e-4,

        "weight_decay": 1e-4,

        "dropout": 0.40,
        "label_smoothing": 0.03,

        "scheduler_patience": 3,
        "scheduler_factor": 0.5,

        "min_backbone_lr": 1e-6,
        "min_head_lr": 1e-5

    },

    "convnext_tiny_fpn": {

        "batch_size": 8,
        "accumulation_steps": 2,

        "max_epochs": 70,
        "early_stopping_patience": 12,

        "backbone_lr": 1e-5,
        "head_lr": 2e-4,

        "weight_decay": 0.05,

        "dropout": 0.30,
        "label_smoothing": 0.08,

        "scheduler_patience": 4,
        "scheduler_factor": 0.5,

        "min_backbone_lr": 5e-7,
        "min_head_lr": 5e-6

    },

    "efficientnet_b0_fpn": {

        "batch_size": 24,
        "accumulation_steps": 1,

        "max_epochs": 60,
        "early_stopping_patience": 10,

        "backbone_lr": 8e-5,
        "head_lr": 4e-4,

        "weight_decay": 1e-5,

        "dropout": 0.25,
        "label_smoothing": 0.05,

        "scheduler_patience": 3,
        "scheduler_factor": 0.5,

        "min_backbone_lr": 1e-6,
        "min_head_lr": 1e-5

    }

}

MODEL_ORDER = list(MODEL_CONFIGS.keys())

MODEL_ROOT = (
    PROJECT_ROOT /
    "models" /
    "experiment19_fpn_backbones"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# AMP helpers
# ------------------------------------------------------------

def get_scaler():

    try:

        return torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP
        )

    except TypeError:

        return torch.cuda.amp.GradScaler(
            enabled=USE_AMP
        )


def autocast_context():

    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=USE_AMP
    )


# ------------------------------------------------------------
# Train one epoch
# ------------------------------------------------------------

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    accumulation_steps
):

    model.train()

    total_loss = 0.0
    total_images = 0

    optimizer.zero_grad(
        set_to_none=True
    )

    for step, (images, labels, _) in enumerate(loader):

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        # Correctly handle the final incomplete
        # gradient accumulation group.

        group_start = (
            step // accumulation_steps
        ) * accumulation_steps

        group_size = min(
            accumulation_steps,
            len(loader) - group_start
        )

        with autocast_context():

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            scaled_loss = loss / group_size

        scaler.scale(
            scaled_loss
        ).backward()

        should_update = (

            (step + 1) % accumulation_steps == 0

            or step + 1 == len(loader)

        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        total_loss += (
            loss.item() * len(labels)
        )

        total_images += len(labels)

    return total_loss / total_images


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

@torch.inference_mode()
def validate_model(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    total_images = 0

    actual = []
    predicted = []

    for images, labels, _ in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        labels = labels.to(DEVICE)

        with autocast_context():

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        predictions = logits.argmax(dim=1)

        total_loss += (
            loss.item() * len(labels)
        )

        total_images += len(labels)

        actual.extend(
            labels.cpu().numpy()
        )

        predicted.extend(
            predictions.cpu().numpy()
        )

    actual = np.asarray(actual)

    predicted = np.asarray(predicted)

    return {

        "val_loss": total_loss / total_images,

        "val_accuracy": accuracy_score(
            actual,
            predicted
        ),

        "val_macro_f1": f1_score(
            actual,
            predicted,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        )

    }


# ------------------------------------------------------------
# Train complete FPN architecture
# ------------------------------------------------------------

def train_fpn_model(architecture):

    cfg = MODEL_CONFIGS[architecture]

    seed_everything(SEED)

    train_loader, val_loader, _ = make_loaders(
        cfg["batch_size"]
    )

    model = create_fpn_model(
        architecture,
        pretrained=True,
        dropout=cfg["dropout"]
    )

    model = model.to(
        DEVICE,
        memory_format=torch.channels_last
    )

    # Separate backbone and FPN/head parameters.

    backbone_parameters = list(
        model.backbone.parameters()
    )

    head_parameters = list(
        model.fpn_head.parameters()
    )

    optimizer = torch.optim.AdamW(

        [

            {
                "params": backbone_parameters,
                "lr": cfg["backbone_lr"]
            },

            {
                "params": head_parameters,
                "lr": cfg["head_lr"]
            }

        ],

        weight_decay=cfg["weight_decay"]

    )

    criterion = nn.CrossEntropyLoss(

        weight=CLASS_WEIGHTS,

        label_smoothing=cfg["label_smoothing"]

    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=cfg["scheduler_factor"],

        patience=cfg["scheduler_patience"],

        min_lr=[
            cfg["min_backbone_lr"],
            cfg["min_head_lr"]
        ]

    )

    scaler = get_scaler()

    run_dir = MODEL_ROOT / architecture

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    best_path = (
        run_dir / "best_model.pth"
    )

    history = []

    best_f1 = -1.0
    best_loss = float("inf")

    bad_epochs = 0

    start_time = time.time()

    torch.cuda.reset_peak_memory_stats()

    print("\n" + "=" * 75)
    print("TRAINING:", architecture)
    print("=" * 75)

    for epoch in range(
        1,
        cfg["max_epochs"] + 1
    ):

        epoch_start = time.time()

        train_loss = train_one_epoch(

            model,

            train_loader,

            optimizer,

            criterion,

            scaler,

            cfg["accumulation_steps"]

        )

        val_metrics = validate_model(

            model,

            val_loader,

            criterion

        )

        current_f1 = val_metrics["val_macro_f1"]

        current_loss = val_metrics["val_loss"]

        scheduler.step(current_f1)

        improved = (

            current_f1 > best_f1 + 1e-5

            or (

                abs(current_f1 - best_f1) <= 1e-5

                and current_loss < best_loss

            )

        )

        if improved:

            best_f1 = current_f1
            best_loss = current_loss

            bad_epochs = 0

            checkpoint = {

                "architecture": architecture,

                "model_state_dict": {

                    k: v.detach().cpu().clone()

                    for k, v in model.state_dict().items()

                },

                "class_names": [
                    class_names[i]
                    for i in range(NUM_CLASSES)
                ],

                "num_classes": NUM_CLASSES,

                "image_size": IMAGE_SIZE,

                "imagenet_mean": list(IMAGENET_MEAN),

                "imagenet_std": list(IMAGENET_STD),

                "best_epoch": epoch,

                "best_val_macro_f1": best_f1,

                "best_val_loss": best_loss,

                "config": cfg

            }

            torch.save(
                checkpoint,
                best_path
            )

        else:

            bad_epochs += 1

        epoch_seconds = (
            time.time() - epoch_start
        )

        history.append({

            "epoch": epoch,

            "train_loss": train_loss,

            "val_loss": current_loss,

            "val_accuracy": val_metrics["val_accuracy"],

            "val_macro_f1": current_f1,

            "backbone_lr": optimizer.param_groups[0]["lr"],

            "head_lr": optimizer.param_groups[1]["lr"],

            "epoch_seconds": epoch_seconds

        })

        pd.DataFrame(history).to_csv(
            run_dir / "training_history.csv",
            index=False
        )

        print(

            f"Epoch {epoch:02d}/{cfg['max_epochs']} | "

            f"Train Loss {train_loss:.4f} | "

            f"Val Loss {current_loss:.4f} | "

            f"Val Accuracy {val_metrics['val_accuracy']:.4f} | "

            f"Val Macro F1 {current_f1:.4f} | "

            f"Best F1 {best_f1:.4f} | "

            f"Time {epoch_seconds:.1f}s"

        )

        if bad_epochs >= cfg["early_stopping_patience"]:

            print("\nEarly stopping activated.")

            break

    training_seconds = (
        time.time() - start_time
    )

    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    # Save configuration separately.

    config_output = {

        **cfg,

        "architecture": architecture,

        "optimizer": "AdamW",

        "scheduler": "ReduceLROnPlateau",

        "loss": "Weighted CrossEntropy",

        "pretrained": True,

        "image_size": IMAGE_SIZE,

        "seed": SEED,

        "best_val_macro_f1": best_f1,

        "best_val_loss": best_loss,

        "training_minutes": training_seconds / 60,

        "peak_gpu_memory_gb": peak_memory_gb

    }

    with open(
        run_dir / "config.json",
        "w"
    ) as f:

        json.dump(
            config_output,
            f,
            indent=4
        )

    print("\nCOMPLETED:", architecture)

    print("Best validation F1:", best_f1)

    print("Training minutes:", round(training_seconds / 60, 2))

    print("Peak GPU memory:", round(peak_memory_gb, 2), "GB")

    print("Checkpoint:", best_path)

    del model
    del train_loader, val_loader

    torch.cuda.empty_cache()

    return {

        "architecture": architecture,

        "checkpoint": str(best_path),

        "best_val_macro_f1": best_f1,

        "training_minutes": training_seconds / 60,

        "peak_gpu_memory_gb": peak_memory_gb,

        "best_epoch": int(
            max(history, key=lambda x: x["val_macro_f1"])["epoch"]
        )

    }


print("Individual training configurations and functions ready.")


In [ ]:

# ============================================================
# TRAIN DENSENET121 + FPN
# TRAIN CONVNEXT-TINY + FPN
# TRAIN EFFICIENTNET-B0 + FPN
# ============================================================

training_results = []

for architecture in MODEL_ORDER:

    print("\n" + "#" * 80)

    print("STARTING EXPERIMENT:", architecture)

    print("#" * 80)

    result = train_fpn_model(
        architecture
    )

    training_results.append(result)

    pd.DataFrame(
        training_results
    ).to_csv(
        MODEL_ROOT / "training_summary.csv",
        index=False
    )

    print("\nExperiment completed:", architecture)

print("\n" + "=" * 80)
print("ALL THREE FPN MODELS TRAINED")
print("=" * 80)

training_summary_df = pd.DataFrame(
    training_results
)

display(training_summary_df)


In [ ]:

# ============================================================
# ADVANCED EVALUATION FUNCTIONS
# ============================================================

def calculate_ece(y_true, probabilities, bins=15):

    confidence = probabilities.max(axis=1)

    predicted = probabilities.argmax(axis=1)

    correct = (
        predicted == y_true
    ).astype(float)

    edges = np.linspace(0, 1, bins + 1)

    ece = 0.0

    for i in range(bins):

        if i == bins - 1:

            mask = (
                (confidence >= edges[i])
                & (confidence <= edges[i + 1])
            )

        else:

            mask = (
                (confidence >= edges[i])
                & (confidence < edges[i + 1])
            )

        if mask.sum() == 0:
            continue

        accuracy = correct[mask].mean()

        avg_confidence = confidence[mask].mean()

        ece += (
            mask.sum() / len(y_true)
        ) * abs(
            accuracy - avg_confidence
        )

    return float(ece)


def calculate_specificity(cm):

    values = []

    total = cm.sum()

    for i in range(NUM_CLASSES):

        tp = cm[i, i]

        fn = cm[i, :].sum() - tp

        fp = cm[:, i].sum() - tp

        tn = total - tp - fn - fp

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        values.append(specificity)

    return np.asarray(values)


# ------------------------------------------------------------
# Prediction collection
# ------------------------------------------------------------

@torch.inference_mode()
def collect_predictions(model, loader):

    model.eval()

    # GPU warm-up

    warm_images, _, _ = next(iter(loader))

    warm_images = warm_images.to(
        DEVICE,
        non_blocking=True,
        memory_format=torch.channels_last
    )

    with autocast_context():

        model(warm_images)

    torch.cuda.synchronize()

    actual = []
    predicted = []
    probabilities = []
    indices = []

    batch_latency = []

    total_forward_seconds = 0.0

    for images, labels, batch_indices in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        torch.cuda.synchronize()

        start = time.perf_counter()

        with autocast_context():

            logits = model(images)

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        total_forward_seconds += elapsed

        batch_latency.append(
            elapsed / len(images) * 1000
        )

        probs = torch.softmax(
            logits.float(),
            dim=1
        )

        preds = probs.argmax(dim=1)

        actual.extend(labels.numpy())

        predicted.extend(
            preds.cpu().numpy()
        )

        probabilities.extend(
            probs.cpu().numpy()
        )

        indices.extend(
            batch_indices.numpy()
        )

    return {

        "y_true": np.asarray(actual),

        "y_pred": np.asarray(predicted),

        "y_prob": np.asarray(probabilities),

        "indices": np.asarray(indices),

        "forward_seconds": total_forward_seconds,

        "batch_latency_ms": np.asarray(batch_latency)

    }


# ------------------------------------------------------------
# Full evaluation
# ------------------------------------------------------------

def evaluate_predictions(
    prediction,
    metadata,
    model_name,
    checkpoint_path,
    training_minutes
):

    y_true = prediction["y_true"]

    y_pred = prediction["y_pred"]

    y_prob = prediction["y_prob"]

    indices = prediction["indices"]

    one_hot = np.eye(NUM_CLASSES)[y_true]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    report = classification_report(

        y_true,

        y_pred,

        labels=list(range(NUM_CLASSES)),

        target_names=[
            class_names[i]
            for i in range(NUM_CLASSES)
        ],

        output_dict=True,

        zero_division=0

    )

    specificity = calculate_specificity(cm)

    roc_auc = roc_auc_score(

        y_true,

        y_prob,

        labels=list(range(NUM_CLASSES)),

        multi_class="ovr",

        average="macro"

    )

    pr_auc = average_precision_score(
        one_hot,
        y_prob,
        average="macro"
    )

    brier = np.mean(
        np.sum(
            (y_prob - one_hot) ** 2,
            axis=1
        )
    )

    ece = calculate_ece(
        y_true,
        y_prob
    )

    top2 = np.argsort(
        y_prob,
        axis=1
    )[:, -2:]

    top2_accuracy = np.mean(
        np.any(
            top2 == y_true[:, None],
            axis=1
        )
    )

    # --------------------------------------------------------
    # Per-class results
    # --------------------------------------------------------

    per_class = []

    for i in range(NUM_CLASSES):

        name = class_names[i]

        per_class.append({

            "model": model_name,

            "class_id": i,

            "class_name": name,

            "precision": report[name]["precision"],

            "recall": report[name]["recall"],

            "f1_score": report[name]["f1-score"],

            "specificity": specificity[i],

            "support": report[name]["support"]

        })

    per_class_df = pd.DataFrame(
        per_class
    )

    # --------------------------------------------------------
    # Fabric-wise results
    # --------------------------------------------------------

    metadata_used = (
        metadata.iloc[indices]
        .reset_index(drop=True)
    )

    fabric_data = pd.DataFrame({

        "fabric": metadata_used["folder_1"],

        "actual": y_true,

        "predicted": y_pred

    })

    fabric_results = []

    for fabric, group in fabric_data.groupby("fabric"):

        fabric_results.append({

            "model": model_name,

            "fabric": fabric,

            "image_count": len(group),

            "accuracy": accuracy_score(
                group["actual"],
                group["predicted"]
            ),

            "macro_f1": f1_score(

                group["actual"],

                group["predicted"],

                labels=list(range(NUM_CLASSES)),

                average="macro",

                zero_division=0

            )

        })

    fabric_df = pd.DataFrame(
        fabric_results
    )

    inference_seconds = (
        prediction["forward_seconds"]
    )

    throughput = (
        len(y_true) / inference_seconds
        if inference_seconds > 0
        else 0
    )

    latency = prediction["batch_latency_ms"]

    model_size_mb = (
        Path(checkpoint_path).stat().st_size
        / (1024**2)
    )

    parameter_count = sum(
        p.numel()
        for p in load_model_for_count(model_name).parameters()
    )

    metrics = {

        "model": model_name,

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),

        "mcc": matthews_corrcoef(
            y_true,
            y_pred
        ),

        "cohen_kappa": cohen_kappa_score(
            y_true,
            y_pred
        ),

        "macro_roc_auc": roc_auc,

        "macro_pr_auc": pr_auc,

        "log_loss": log_loss(
            y_true,
            y_prob,
            labels=list(range(NUM_CLASSES))
        ),

        "multiclass_brier_score": brier,

        "ece": ece,

        "top2_accuracy": top2_accuracy,

        "training_minutes": training_minutes,

        "inference_seconds": inference_seconds,

        "latency_ms_per_image": (
            inference_seconds / len(y_true) * 1000
        ),

        "p95_batch_latency_ms_per_image": (
            np.percentile(latency, 95)
        ),

        "throughput_images_per_second": throughput,

        "parameters": parameter_count,

        "model_size_mb": model_size_mb,

        "worst_fabric_macro_f1": fabric_df["macro_f1"].min()

    }

    return {

        "metrics": metrics,

        "per_class": per_class_df,

        "per_fabric": fabric_df,

        "confusion_matrix": cm,

        "predictions": prediction

    }


# Helper for parameter counting

def load_model_for_count(architecture):

    cfg = MODEL_CONFIGS[architecture]

    model = create_fpn_model(
        architecture,
        pretrained=False,
        dropout=cfg["dropout"]
    )

    return model


print("Advanced evaluation functions ready.")


In [ ]:

# ============================================================
# LOAD CHECKPOINTS AND EVALUATE
# ============================================================

EVALUATION_ROOT = (
    PROJECT_ROOT /
    "reports" /
    "model_evaluation" /
    "experiment19_fpn_backbones"
)

EVALUATION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def load_trained_fpn(architecture):

    cfg = MODEL_CONFIGS[architecture]

    checkpoint_path = (
        MODEL_ROOT /
        architecture /
        "best_model.pth"
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: {checkpoint_path}"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model = create_fpn_model(

        architecture,

        pretrained=False,

        dropout=cfg["dropout"]

    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(
        DEVICE,
        memory_format=torch.channels_last
    )

    model.eval()

    return model, checkpoint_path


evaluation_results = {}

all_metrics = []
all_class_metrics = []
all_fabric_metrics = []

for architecture in MODEL_ORDER:

    print("\n" + "=" * 75)

    print("EVALUATING:", architecture)

    print("=" * 75)

    model, checkpoint_path = load_trained_fpn(
        architecture
    )

    training_minutes = next(
        r["training_minutes"]
        for r in training_results
        if r["architecture"] == architecture
    )

    batch_size = MODEL_CONFIGS[architecture]["batch_size"]

    model_results = {}

    for split_name, metadata in [

        ("validation", val_df),

        ("test", test_df)

    ]:

        loader = make_eval_loader(
            metadata,
            batch_size
        )

        prediction = collect_predictions(
            model,
            loader
        )

        result = evaluate_predictions(

            prediction,

            metadata,

            architecture,

            checkpoint_path,

            training_minutes

        )

        model_results[split_name] = result

        metric_row = result["metrics"].copy()

        metric_row["split"] = split_name

        all_metrics.append(metric_row)

        all_class_metrics.append(
            result["per_class"].assign(
                split=split_name
            )
        )

        all_fabric_metrics.append(
            result["per_fabric"].assign(
                split=split_name
            )
        )

        # Save confusion matrix

        cm_df = pd.DataFrame(

            result["confusion_matrix"],

            index=[
                class_names[i]
                for i in range(NUM_CLASSES)
            ],

            columns=[
                class_names[i]
                for i in range(NUM_CLASSES)
            ]

        )

        cm_df.to_csv(
            EVALUATION_ROOT /
            f"{architecture}_{split_name}_confusion_matrix.csv"
        )

        # Normalized confusion matrix

        normalized_cm = (
            result["confusion_matrix"]
            / result["confusion_matrix"].sum(
                axis=1,
                keepdims=True
            )
        )

        pd.DataFrame(
            normalized_cm,
            index=cm_df.index,
            columns=cm_df.columns
        ).to_csv(
            EVALUATION_ROOT /
            f"{architecture}_{split_name}_normalized_confusion.csv"
        )

        # Save predictions

        predictions_df = (
            metadata.iloc[
                prediction["indices"]
            ]
            .reset_index(drop=True)
            .copy()
        )

        predictions_df["predicted_class_id"] = (
            prediction["y_pred"]
        )

        predictions_df["predicted_class"] = [
            class_names[i]
            for i in prediction["y_pred"]
        ]

        predictions_df["confidence"] = (
            prediction["y_prob"].max(axis=1)
        )

        predictions_df["actual_class_probability"] = (
            prediction["y_prob"][
                np.arange(len(prediction["y_true"])),
                prediction["y_true"]
            ]
        )

        predictions_df["correct"] = (
            prediction["y_true"]
            == prediction["y_pred"]
        )

        predictions_df.to_csv(
            EVALUATION_ROOT /
            f"{architecture}_{split_name}_predictions.csv",
            index=False
        )

        print(
            split_name,
            "| Accuracy:",
            round(result["metrics"]["accuracy"], 5),

            "| Macro F1:",
            round(result["metrics"]["macro_f1"], 5),

            "| Worst fabric F1:",
            round(
                result["metrics"]["worst_fabric_macro_f1"],
                5
            )
        )

        del loader

    evaluation_results[architecture] = model_results

    del model

    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Save consolidated reports
# ------------------------------------------------------------

all_metrics_df = pd.DataFrame(all_metrics)

all_class_metrics_df = pd.concat(
    all_class_metrics,
    ignore_index=True
)

all_fabric_metrics_df = pd.concat(
    all_fabric_metrics,
    ignore_index=True
)

all_metrics_df.to_csv(
    EVALUATION_ROOT / "all_metrics.csv",
    index=False
)

all_class_metrics_df.to_csv(
    EVALUATION_ROOT / "all_per_class_metrics.csv",
    index=False
)

all_fabric_metrics_df.to_csv(
    EVALUATION_ROOT / "all_fabric_metrics.csv",
    index=False
)

print("\nALL THREE MODELS EVALUATED.")

display(
    all_metrics_df[
        [
            "model",
            "split",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
            "macro_roc_auc",
            "macro_pr_auc",
            "ece",
            "worst_fabric_macro_f1",
            "throughput_images_per_second"
        ]
    ]
)


In [ ]:

# ============================================================
# ARCHITECTURE COMPARISON
# ============================================================

validation_ranking = (

    all_metrics_df[
        all_metrics_df["split"] == "validation"
    ]

    .sort_values(
        [
            "macro_f1",
            "worst_fabric_macro_f1",
            "balanced_accuracy"
        ],
        ascending=False
    )

    .reset_index(drop=True)

)

print("=" * 80)
print("VALIDATION RANKING - NEW FPN ARCHITECTURES")
print("=" * 80)

display(
    validation_ranking[
        [
            "model",
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "balanced_accuracy",
            "worst_fabric_macro_f1",
            "macro_roc_auc",
            "macro_pr_auc",
            "ece",
            "training_minutes",
            "parameters",
            "model_size_mb",
            "latency_ms_per_image"
        ]
    ]
)

# ------------------------------------------------------------
# Historical ResNet18 + FPN reference
# ------------------------------------------------------------

previous_resnet_fpn = {

    "model": "ResNet18 + FPN (existing)",

    "accuracy": 0.996377,

    "macro_f1": 0.996673,

    "weighted_f1": 0.996378,

    "parameters": 12024395,

    "training_minutes": 7.114116,

    "inference_seconds": 1.842069

}

print("\nEXISTING RESNET18 + FPN REFERENCE")

display(
    pd.DataFrame([previous_resnet_fpn])
)

# ------------------------------------------------------------
# Per-class comparison
# ------------------------------------------------------------

val_class_df = all_class_metrics_df[
    all_class_metrics_df["split"] == "validation"
]

class_comparison = val_class_df.pivot(

    index="class_name",

    columns="model",

    values="f1_score"

)

print("\nPER-CLASS F1 COMPARISON")

display(
    class_comparison.style.format("{:.4f}")
)

# ------------------------------------------------------------
# Fabric-wise comparison
# ------------------------------------------------------------

val_fabric_df = all_fabric_metrics_df[
    all_fabric_metrics_df["split"] == "validation"
]

fabric_comparison = val_fabric_df.pivot(

    index="fabric",

    columns="model",

    values="macro_f1"

)

print("\nFABRIC-WISE MACRO F1")

display(
    fabric_comparison.style.format("{:.4f}")
)

# ------------------------------------------------------------
# Provisional winner
# ------------------------------------------------------------

provisional_winner = (
    validation_ranking.iloc[0]["model"]
)

print("\nPROVISIONAL WINNER:", provisional_winner)

# ------------------------------------------------------------
# Plot comparison
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

plt.bar(
    validation_ranking["model"],
    validation_ranking["macro_f1"]
)

plt.ylabel("Validation Macro F1")

plt.title(
    "FPN Architecture Comparison"
)

plt.xticks(rotation=20)

plt.tight_layout()

plt.show()

# Save ranking

validation_ranking.to_csv(
    EVALUATION_ROOT / "validation_architecture_ranking.csv",
    index=False
)


In [ ]:

# ============================================================
# ERROR ANALYSIS
# ============================================================

for architecture in MODEL_ORDER:

    result = evaluation_results[architecture]["test"]

    prediction = result["predictions"]

    errors = np.where(
        prediction["y_true"] != prediction["y_pred"]
    )[0]

    error_df = (
        test_df.iloc[
            prediction["indices"][errors]
        ]
        .reset_index(drop=True)
        .copy()
    )

    error_df["actual_class"] = [
        class_names[i]
        for i in prediction["y_true"][errors]
    ]

    error_df["predicted_class"] = [
        class_names[i]
        for i in prediction["y_pred"][errors]
    ]

    error_df["confidence"] = (
        prediction["y_prob"][errors].max(axis=1)
    )

    error_df["actual_probability"] = (
        prediction["y_prob"][
            errors,
            prediction["y_true"][errors]
        ]
    )

    error_df = error_df.sort_values(
        "confidence",
        ascending=False
    )

    error_df.to_csv(
        EVALUATION_ROOT /
        f"{architecture}_incorrect_predictions.csv",
        index=False
    )

    print("\n", architecture)

    print("Incorrect predictions:", len(error_df))

    display(
        error_df[
            [
                "file_name",
                "folder_1",
                "actual_class",
                "predicted_class",
                "confidence"
            ]
        ].head(20)
    )

print("\nERROR ANALYSIS SAVED.")


In [ ]:

# ============================================================
# S3 MODEL BACKUP
# ============================================================

S3_MODEL_PREFIX = (
    "model-artifacts/"
    "stitching-defect-detection/"
    "experiment19_fpn_backbones/"
)

files_to_upload = [

    p for p in MODEL_ROOT.rglob("*")

    if p.is_file()

]

files_to_upload += [

    p for p in EVALUATION_ROOT.rglob("*")

    if p.is_file()

]

# Remove duplicate paths

files_to_upload = list(
    dict.fromkeys(files_to_upload)
)

upload_records = []

for path in tqdm(
    files_to_upload,
    desc="Uploading models and reports"
):

    if path.is_relative_to(MODEL_ROOT):

        relative = path.relative_to(
            MODEL_ROOT
        )

        prefix = "models/"

    else:

        relative = path.relative_to(
            EVALUATION_ROOT
        )

        prefix = "reports/"

    s3_key = (
        S3_MODEL_PREFIX
        + prefix
        + relative.as_posix()
    )

    s3.upload_file(
        str(path),
        BUCKET_NAME,
        s3_key
    )

    upload_records.append({

        "file": str(path),

        "s3_key": s3_key,

        "size_mb": path.stat().st_size / 1024**2

    })

upload_manifest = pd.DataFrame(
    upload_records
)

upload_manifest.to_csv(
    EVALUATION_ROOT / "s3_upload_manifest.csv",
    index=False
)

print("S3 BACKUP COMPLETED")

print("Uploaded files:", len(upload_manifest))

print("S3 prefix:", S3_MODEL_PREFIX)

display(
    upload_manifest.head(20)
)


In [ ]:

# ============================================================
# VERIFY ENHANCED S3 DATASET
# ============================================================

from pathlib import PurePosixPath

assert "s3" in globals()
assert "BUCKET_NAME" in globals()

def list_s3_prefix_sizes(prefix):

    paginator = s3.get_paginator(
        "list_objects_v2"
    )

    result = {}

    for page in paginator.paginate(
        Bucket=BUCKET_NAME,
        Prefix=prefix
    ):

        for obj in page.get("Contents", []):

            result[obj["Key"]] = obj["Size"]

    return result


enhanced_df = split_df.copy()

variant_sizes = {}

for variant in ENHANCED_VARIANTS:

    prefix = (
        f"processed-data/exp17_multiview/{variant}/"
    )

    inventory = list_s3_prefix_sizes(prefix)

    expected_keys = [

        prefix +
        PurePosixPath(key).relative_to("data").as_posix()

        for key in split_df["s3_key"]

    ]

    assert len(expected_keys) == 14565

    assert len(set(expected_keys)) == 14565

    missing = [
        key for key in expected_keys
        if key not in inventory
    ]

    assert len(missing) == 0, (
        f"{variant}: missing {len(missing)} images"
    )

    assert all(
        inventory[key] > 0
        for key in expected_keys
    )

    column = f"{variant}_s3_key"

    enhanced_df[column] = expected_keys

    variant_sizes[variant] = {

        key: inventory[key]

        for key in expected_keys

    }

    total_gb = (
        sum(variant_sizes[variant].values())
        / 1024**3
    )

    print("\n", variant)

    print("Expected:", len(expected_keys))

    print("Available:", len(inventory))

    print("Missing:", len(missing))

    print("Storage (GB):", round(total_gb, 3))


assert enhanced_df[
    [
        "lab_clahe_s3_key",
        "bilateral_clahe_s3_key"
    ]
].notna().all().all()

print("\nBOTH ENHANCED DATASETS VERIFIED.")


In [ ]:

# ============================================================
# DOWNLOAD ENHANCED IMAGES
# ============================================================

from concurrent.futures import ThreadPoolExecutor
import gc

ENHANCED_CACHE_ROOT = (
    PROJECT_ROOT /
    "data" /
    "training_cache" /
    "enhanced"
)

ENHANCED_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def enhanced_local_path(variant, s3_key):

    parts = PurePosixPath(s3_key).parts

    assert parts[:3] == (
        "processed-data",
        "exp17_multiview",
        variant
    )

    return (
        ENHANCED_CACHE_ROOT /
        variant /
        Path(*parts[3:])
    )


required_bytes = sum(
    sum(sizes.values())
    for sizes in variant_sizes.values()
)

available_bytes = shutil.disk_usage(
    ENHANCED_CACHE_ROOT
).free

print(
    "Required storage (GB):",
    round(required_bytes / 1024**3, 3)
)

print(
    "Available storage (GB):",
    round(available_bytes / 1024**3, 3)
)

assert available_bytes > required_bytes * 1.15, (
    "Insufficient storage for both enhanced datasets."
)


def download_enhanced_image(task):

    key, path = task

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    s3.download_file(
        BUCKET_NAME,
        key,
        str(path)
    )


for variant in ENHANCED_VARIANTS:

    sizes = variant_sizes[variant]

    tasks = []

    for key, expected_size in sizes.items():

        path = enhanced_local_path(
            variant,
            key
        )

        if (
            not path.exists()
            or path.stat().st_size != expected_size
        ):

            tasks.append((key, path))

    print(
        f"\n{variant}: downloading {len(tasks)} images"
    )

    if tasks:

        with ThreadPoolExecutor(
            max_workers=6
        ) as executor:

            list(
                tqdm(
                    executor.map(
                        download_enhanced_image,
                        tasks
                    ),
                    total=len(tasks),
                    desc=variant
                )
            )

    # Verify every local file and its exact size.

    for key, expected_size in sizes.items():

        path = enhanced_local_path(
            variant,
            key
        )

        assert path.exists()

        assert path.stat().st_size == expected_size


print("\nENHANCED LOCAL CACHE VERIFIED")

for variant in ENHANCED_VARIANTS:

    print(
        variant,
        "images:",
        len(variant_sizes[variant])
    )


In [ ]:

# ============================================================
# ENHANCED DATASET
# ============================================================

class EnhancedStitchingDataset(Dataset):

    def __init__(
        self,
        dataframe,
        variant,
        training=False
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.variant = variant

        self.training = training

        self.key_column = (
            f"{variant}_s3_key"
        )

        self.transform = (
            train_transform
            if training
            else evaluation_transform
        )

        self.paths = [

            enhanced_local_path(
                variant,
                key
            )

            for key in self.df[self.key_column]

        ]

        self.labels = (
            self.df["class_id"]
            .astype(int)
            .tolist()
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        with Image.open(
            self.paths[index]
        ) as image:

            image = ImageOps.exif_transpose(
                image
            ).convert("RGB")

        image = self.transform(image)

        return (

            image,

            self.labels[index],

            index

        )


def make_enhanced_loaders(
    variant,
    batch_size
):

    train_data = enhanced_df[
        enhanced_df["split"] == "train"
    ].reset_index(drop=True)

    val_data = enhanced_df[
        enhanced_df["split"] == "validation"
    ].reset_index(drop=True)

    test_data = enhanced_df[
        enhanced_df["split"] == "test"
    ].reset_index(drop=True)

    train_dataset = EnhancedStitchingDataset(
        train_data,
        variant,
        training=True
    )

    val_dataset = EnhancedStitchingDataset(
        val_data,
        variant,
        training=False
    )

    test_dataset = EnhancedStitchingDataset(
        test_data,
        variant,
        training=False
    )

    generator = torch.Generator()

    generator.manual_seed(SEED)

    common = {

        "batch_size": batch_size,

        "num_workers": NUM_WORKERS,

        "pin_memory": True,

        "persistent_workers": True

    }

    train_loader = DataLoader(

        train_dataset,

        shuffle=True,

        generator=generator,

        **common

    )

    val_loader = DataLoader(

        val_dataset,

        shuffle=False,

        **common

    )

    test_loader = DataLoader(

        test_dataset,

        shuffle=False,

        **common

    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


print("Enhanced dataset and loader functions ready.")

for variant in ENHANCED_VARIANTS:

    print("\n", variant)

    print("Train:", len(train_df))

    print("Validation:", len(val_df))

    print("Test:", len(test_df))


In [ ]:

# ============================================================
# TRAIN ENHANCED EFFICIENTNET-B0 + FPN
# ============================================================

ENHANCED_MODEL_ROOT = (
    PROJECT_ROOT /
    "models" /
    "experiment20_enhanced_fpn"
)

ENHANCED_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def train_enhanced_variant(variant):

    cfg = ENHANCED_CONFIG.copy()

    seed_everything(SEED)

    train_loader, val_loader, test_loader = (
        make_enhanced_loaders(
            variant,
            cfg["batch_size"]
        )
    )

    # ImageNet-pretrained backbone.
    # New FPN and classification head.

    model = create_fpn_model(

        SELECTED_ARCHITECTURE,

        pretrained=True,

        dropout=cfg["dropout"]

    )

    model = model.to(
        DEVICE,
        memory_format=torch.channels_last
    )

    optimizer = torch.optim.AdamW(

        [

            {
                "params": model.backbone.parameters(),
                "lr": cfg["backbone_lr"]
            },

            {
                "params": model.fpn_head.parameters(),
                "lr": cfg["head_lr"]
            }

        ],

        weight_decay=cfg["weight_decay"]

    )

    criterion = nn.CrossEntropyLoss(

        weight=CLASS_WEIGHTS,

        label_smoothing=cfg["label_smoothing"]

    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=cfg["scheduler_factor"],

        patience=cfg["scheduler_patience"],

        min_lr=[
            cfg["min_backbone_lr"],
            cfg["min_head_lr"]
        ]

    )

    scaler = get_scaler()

    run_dir = (
        ENHANCED_MODEL_ROOT /
        variant
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    best_path = (
        run_dir /
        "best_model.pth"
    )

    history = []

    best_f1 = -1.0

    best_loss = float("inf")

    bad_epochs = 0

    start_time = time.time()

    torch.cuda.reset_peak_memory_stats()

    print("\n" + "=" * 75)

    print("TRAINING:", variant)

    print("Architecture:", SELECTED_ARCHITECTURE)

    print("=" * 75)

    for epoch in range(
        1,
        cfg["max_epochs"] + 1
    ):

        epoch_start = time.time()

        train_loss = train_one_epoch(

            model,

            train_loader,

            optimizer,

            criterion,

            scaler,

            cfg["accumulation_steps"]

        )

        val_metrics = validate_model(

            model,

            val_loader,

            criterion

        )

        current_f1 = val_metrics["val_macro_f1"]

        current_loss = val_metrics["val_loss"]

        scheduler.step(current_f1)

        improved = (

            current_f1 > best_f1 + 1e-5

            or (

                abs(current_f1 - best_f1) <= 1e-5

                and current_loss < best_loss

            )

        )

        if improved:

            best_f1 = current_f1

            best_loss = current_loss

            bad_epochs = 0

            checkpoint = {

                "architecture": SELECTED_ARCHITECTURE,

                "variant": variant,

                "model_state_dict": {

                    k: v.detach().cpu().clone()

                    for k, v in model.state_dict().items()

                },

                "class_names": [

                    class_names[i]

                    for i in range(NUM_CLASSES)

                ],

                "num_classes": NUM_CLASSES,

                "image_size": IMAGE_SIZE,

                "imagenet_mean": list(IMAGENET_MEAN),

                "imagenet_std": list(IMAGENET_STD),

                "best_epoch": epoch,

                "best_val_macro_f1": best_f1,

                "best_val_loss": best_loss,

                "config": cfg

            }

            torch.save(
                checkpoint,
                best_path
            )

        else:

            bad_epochs += 1

        epoch_seconds = (
            time.time() - epoch_start
        )

        history.append({

            "epoch": epoch,

            "train_loss": train_loss,

            "val_loss": current_loss,

            "val_accuracy": val_metrics["val_accuracy"],

            "val_macro_f1": current_f1,

            "backbone_lr": optimizer.param_groups[0]["lr"],

            "head_lr": optimizer.param_groups[1]["lr"],

            "epoch_seconds": epoch_seconds

        })

        pd.DataFrame(history).to_csv(

            run_dir / "training_history.csv",

            index=False

        )

        print(

            f"Epoch {epoch:02d}/{cfg['max_epochs']} | "

            f"Train Loss {train_loss:.4f} | "

            f"Val Loss {current_loss:.4f} | "

            f"Val Accuracy {val_metrics['val_accuracy']:.5f} | "

            f"Val Macro F1 {current_f1:.5f} | "

            f"Best F1 {best_f1:.5f} | "

            f"Time {epoch_seconds:.1f}s"

        )

        if bad_epochs >= cfg["early_stopping_patience"]:

            print("Early stopping activated.")

            break

    training_seconds = (
        time.time() - start_time
    )

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    result = {

        "variant": variant,

        "architecture": SELECTED_ARCHITECTURE,

        "best_val_macro_f1": best_f1,

        "best_val_loss": best_loss,

        "best_epoch": max(
            history,
            key=lambda x: x["val_macro_f1"]
        )["epoch"],

        "training_minutes": training_seconds / 60,

        "peak_gpu_memory_gb": peak_memory,

        "checkpoint": str(best_path)

    }

    with open(
        run_dir / "config.json",
        "w"
    ) as f:

        json.dump(
            {
                **cfg,
                **result
            },
            f,
            indent=4
        )

    pd.DataFrame(history).to_csv(
        run_dir / "training_history.csv",
        index=False
    )

    del model

    del train_loader, val_loader, test_loader

    gc.collect()

    torch.cuda.empty_cache()

    print("\nCOMPLETED:", variant)

    print("Best validation F1:", best_f1)

    print(
        "Training minutes:",
        round(training_seconds / 60, 2)
    )

    print("Checkpoint:", best_path)

    return result


print("Enhanced training function ready.")


In [ ]:

# ============================================================
# RUN EXPERIMENT 20A AND 20B
# ============================================================

enhanced_training_results = []

for variant in ENHANCED_VARIANTS:

    print("\n" + "#" * 80)

    print("STARTING:", variant)

    print("#" * 80)

    result = train_enhanced_variant(
        variant
    )

    enhanced_training_results.append(
        result
    )

    pd.DataFrame(
        enhanced_training_results
    ).to_csv(
        ENHANCED_MODEL_ROOT /
        "enhanced_training_summary.csv",
        index=False
    )

print("\nBOTH ENHANCED MODELS TRAINED")

display(
    pd.DataFrame(
        enhanced_training_results
    )
)


In [ ]:

# ============================================================
# EXPERIMENT 20 - ADVANCED EVALUATION
# ============================================================

ENHANCED_EVALUATION_ROOT = (
    PROJECT_ROOT /
    "reports" /
    "model_evaluation" /
    "experiment20_enhanced_fpn"
)

ENHANCED_EVALUATION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def load_enhanced_model(variant):

    checkpoint_path = (
        ENHANCED_MODEL_ROOT /
        variant /
        "best_model.pth"
    )

    assert checkpoint_path.exists()

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model = create_fpn_model(

        SELECTED_ARCHITECTURE,

        pretrained=False,

        dropout=ENHANCED_CONFIG["dropout"]

    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(
        DEVICE,
        memory_format=torch.channels_last
    )

    model.eval()

    return model, checkpoint_path


def make_enhanced_eval_loader(
    variant,
    metadata,
    batch_size
):

    dataset = EnhancedStitchingDataset(

        metadata,

        variant,

        training=False

    )

    return DataLoader(

        dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=True,

        persistent_workers=True

    )


enhanced_evaluation_results = {}

enhanced_metrics_records = []

enhanced_class_records = []

enhanced_fabric_records = []


for variant in ENHANCED_VARIANTS:

    print("\n" + "=" * 75)

    print("EVALUATING:", variant)

    print("=" * 75)

    model, checkpoint_path = load_enhanced_model(
        variant
    )

    training_minutes = next(

        r["training_minutes"]

        for r in enhanced_training_results

        if r["variant"] == variant

    )

    variant_results = {}

    for split_name in [
        "validation",
        "test"
    ]:

        metadata = (
            enhanced_df[
                enhanced_df["split"] == split_name
            ]
            .reset_index(drop=True)
        )

        loader = make_enhanced_eval_loader(

            variant,

            metadata,

            ENHANCED_CONFIG["batch_size"]

        )

        prediction = collect_predictions(
            model,
            loader
        )

        result = evaluate_predictions(

            prediction,

            metadata,

            SELECTED_ARCHITECTURE,

            checkpoint_path,

            training_minutes

        )

        
# ------------------------------------------------------------
# ADD MODEL, VARIANT AND SPLIT IDENTIFICATION
# ------------------------------------------------------------

result["metrics"]["variant"] = variant

result["metrics"]["model"] = (
    f"{SELECTED_ARCHITECTURE}_{variant}"
)

result["metrics"]["split"] = split_name

result["per_class"]["variant"] = variant
result["per_fabric"]["variant"] = variant

result["per_class"]["split"] = split_name
result["per_fabric"]["split"] = split_name

        )

        result["per_class"]["variant"] = variant

        result["per_fabric"]["variant"] = variant

        result["per_class"]["split"] = split_name

        result["per_fabric"]["split"] = split_name

        variant_results[split_name] = result

        enhanced_metrics_records.append(
            result["metrics"]
        )

        enhanced_class_records.append(
            result["per_class"].copy()
        )

        enhanced_fabric_records.append(
            result["per_fabric"].copy()
        )

        # Confusion matrix

        cm_df = pd.DataFrame(

            result["confusion_matrix"],

            index=[
                class_names[i]
                for i in range(NUM_CLASSES)
            ],

            columns=[
                class_names[i]
                for i in range(NUM_CLASSES)
            ]

        )

        cm_df.to_csv(

            ENHANCED_EVALUATION_ROOT /

            f"{variant}_{split_name}_confusion_matrix.csv"

        )

        # Save individual predictions.

        predictions_df = (
            metadata.iloc[
                prediction["indices"]
            ]
            .reset_index(drop=True)
            .copy()
        )

        predictions_df["predicted_class_id"] = (
            prediction["y_pred"]
        )

        predictions_df["predicted_class"] = [

            class_names[i]

            for i in prediction["y_pred"]

        ]

        predictions_df["confidence"] = (
            prediction["y_prob"].max(axis=1)
        )

        predictions_df["correct"] = (
            prediction["y_true"]
            == prediction["y_pred"]
        )

        predictions_df.to_csv(

            ENHANCED_EVALUATION_ROOT /

            f"{variant}_{split_name}_predictions.csv",

            index=False

        )

        print(

            split_name,

            "| Accuracy:",
            round(result["metrics"]["accuracy"], 6),

            "| Macro F1:",
            round(result["metrics"]["macro_f1"], 6),

            "| ECE:",
            round(result["metrics"]["ece"], 6),

            "| Worst fabric F1:",
            round(
                result["metrics"]["worst_fabric_macro_f1"],
                6
            )

        )

        del loader

    enhanced_evaluation_results[variant] = (
        variant_results
    )

    del model

    gc.collect()

    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Consolidated evaluation reports
# ------------------------------------------------------------

enhanced_metrics_df = pd.DataFrame(
    enhanced_metrics_records
)

enhanced_class_df = pd.concat(
    enhanced_class_records,
    ignore_index=True
)

enhanced_fabric_df = pd.concat(
    enhanced_fabric_records,
    ignore_index=True
)

enhanced_metrics_df.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "enhanced_all_metrics.csv",

    index=False

)

enhanced_class_df.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "enhanced_per_class_metrics.csv",

    index=False

)

enhanced_fabric_df.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "enhanced_fabric_metrics.csv",

    index=False

)

print("\nBOTH ENHANCED MODELS EVALUATED.")

display(
    enhanced_metrics_df[
        [
            "model",
            "variant",
            "split",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
            "macro_roc_auc",
            "macro_pr_auc",
            "ece",
            "worst_fabric_macro_f1",
            "throughput_images_per_second"
        ]
    ]
)


In [ ]:

# ============================================================
# ORIGINAL RGB vs ENHANCED INPUTS
# ============================================================

original_metrics = all_metrics_df[
    all_metrics_df["model"] == SELECTED_ARCHITECTURE
].copy()

original_metrics["variant"] = "original_rgb"

comparison_df = pd.concat(

    [
        original_metrics,
        enhanced_metrics_df
    ],

    ignore_index=True

)

comparison_columns = [

    "variant",
    "split",
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "macro_roc_auc",
    "macro_pr_auc",
    "ece",
    "worst_fabric_macro_f1",
    "throughput_images_per_second"

]

print("=" * 90)

print("ORIGINAL RGB vs ENHANCED INPUTS")

print("=" * 90)

display(
    comparison_df[
        comparison_columns
    ].sort_values(
        ["split", "macro_f1"],
        ascending=[True, False]
    )
)

# ------------------------------------------------------------
# Select winner using validation only
# ------------------------------------------------------------

validation_comparison = (

    comparison_df[
        comparison_df["split"] == "validation"
    ]

    .sort_values(

        [
            "macro_f1",
            "worst_fabric_macro_f1",
            "balanced_accuracy",
            "ece"

        ],

        ascending=[
            False,
            False,
            False,
            True
        ]

    )

    .reset_index(drop=True)

)

print("\nVALIDATION-BASED RANKING")

display(
    validation_comparison[
        comparison_columns
    ]
)

winning_variant = (
    validation_comparison.iloc[0]["variant"]
)

print("\nWINNING INPUT REPRESENTATION:", winning_variant)

# Save the final comparison

comparison_df.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "original_vs_enhanced_comparison.csv",

    index=False

)

validation_comparison.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "validation_input_ranking.csv",

    index=False

)


In [ ]:

# ============================================================
# PER-CLASS AND PER-FABRIC COMPARISON
# ============================================================

original_class = all_class_metrics_df[
    all_class_metrics_df["model"] == SELECTED_ARCHITECTURE
].copy()

original_class["variant"] = "original_rgb"

combined_class = pd.concat(
    [
        original_class,
        enhanced_class_df
    ],
    ignore_index=True
)

original_fabric = all_fabric_metrics_df[
    all_fabric_metrics_df["model"] == SELECTED_ARCHITECTURE
].copy()

original_fabric["variant"] = "original_rgb"

combined_fabric = pd.concat(
    [
        original_fabric,
        enhanced_fabric_df
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Defect-wise F1
# ------------------------------------------------------------

for split_name in [
    "validation",
    "test"
]:

    print("\nDEFECT-WISE F1:", split_name)

    class_pivot = combined_class[
        combined_class["split"] == split_name
    ].pivot(

        index="class_name",

        columns="variant",

        values="f1_score"

    )

    display(
        class_pivot.style.format("{:.5f}")
    )

    class_pivot.to_csv(

        ENHANCED_EVALUATION_ROOT /

        f"{split_name}_defect_comparison.csv"

    )


# ------------------------------------------------------------
# Fabric-wise F1
# ------------------------------------------------------------

for split_name in [
    "validation",
    "test"
]:

    print("\nFABRIC-WISE MACRO F1:", split_name)

    fabric_pivot = combined_fabric[
        combined_fabric["split"] == split_name
    ].pivot(

        index="fabric",

        columns="variant",

        values="macro_f1"

    )

    display(
        fabric_pivot.style.format("{:.5f}")
    )

    fabric_pivot.to_csv(

        ENHANCED_EVALUATION_ROOT /

        f"{split_name}_fabric_comparison.csv"

    )


# ------------------------------------------------------------
# Save combined detailed metrics
# ------------------------------------------------------------

combined_class.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "combined_defect_metrics.csv",

    index=False

)

combined_fabric.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "combined_fabric_metrics.csv",

    index=False

)

print("\nDETAILED COMPARISON COMPLETED.")


In [ ]:

# ============================================================
# EXPERIMENT 20 - S3 BACKUP
# ============================================================

S3_EXPERIMENT_PREFIX = (
    "model-artifacts/"
    "stitching-defect-detection/"
    "experiment20_enhanced_fpn/"
)

upload_files = [

    p for p in ENHANCED_MODEL_ROOT.rglob("*")

    if p.is_file()

]

upload_files += [

    p for p in ENHANCED_EVALUATION_ROOT.rglob("*")

    if p.is_file()

]

upload_manifest_records = []

for path in tqdm(
    upload_files,
    desc="Uploading Experiment 20"
):

    if path.is_relative_to(
        ENHANCED_MODEL_ROOT
    ):

        relative = path.relative_to(
            ENHANCED_MODEL_ROOT
        )

        category = "models"

    else:

        relative = path.relative_to(
            ENHANCED_EVALUATION_ROOT
        )

        category = "reports"

    s3_key = (

        S3_EXPERIMENT_PREFIX

        + category

        + "/"

        + relative.as_posix()

    )

    s3.upload_file(
        str(path),
        BUCKET_NAME,
        s3_key
    )

    upload_manifest_records.append({

        "local_path": str(path),

        "s3_key": s3_key,

        "size_mb": (
            path.stat().st_size / 1024**2
        )

    })


upload_manifest_df = pd.DataFrame(
    upload_manifest_records
)

upload_manifest_df.to_csv(

    ENHANCED_EVALUATION_ROOT /
    "s3_upload_manifest.csv",

    index=False

)

print("\nEXPERIMENT 20 BACKUP COMPLETED")

print("Uploaded files:", len(upload_manifest_df))

print("S3 location:", S3_EXPERIMENT_PREFIX)

display(
    upload_manifest_df.head(20)
)


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 1
# RESNET18 + EXISTING GENERALIZED FPN HEAD
#
# Independent implementation.
# Does NOT modify existing three architectures.
# ============================================================

assert "FPNHead" in globals(), (
    "Existing FPNHead must be defined first."
)

assert "NUM_CLASSES" in globals()

# ============================================================
# RESNET18 + FPN CLASSIFIER
# ============================================================

class R18EnhancedFPNClassifier(nn.Module):

    def __init__(
        self,
        num_classes=11,
        pretrained=True,
        dropout=0.30
    ):

        super().__init__()

        # ----------------------------------------------------
        # ImageNet pretrained ResNet18
        # ----------------------------------------------------

        weights = (
            torchvision.models.ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        self.backbone = torchvision.models.resnet18(
            weights=weights
        )

        self.backbone.fc = nn.Identity()

        # ----------------------------------------------------
        # Same FPN architecture used by other models
        # ----------------------------------------------------

        self.fpn_head = FPNHead(

            input_channels=[
                64,
                128,
                256,
                512
            ],

            num_classes=num_classes,

            fpn_channels=256,

            dropout=dropout

        )

    # ========================================================
    # EXTRACT RESNET18 MULTI-SCALE FEATURES
    # ========================================================

    def extract_features(self, x):

        b = self.backbone

        x = b.conv1(x)

        x = b.bn1(x)

        x = b.relu(x)

        x = b.maxpool(x)

        # Stage 1
        c2 = b.layer1(x)

        # Stage 2
        c3 = b.layer2(c2)

        # Stage 3
        c4 = b.layer3(c3)

        # Stage 4
        c5 = b.layer4(c4)

        return c2, c3, c4, c5

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, x):

        features = self.extract_features(x)

        return self.fpn_head(*features)


# ============================================================
# INDEPENDENT MODEL FACTORY
# ============================================================

def create_resnet18_fpn_model(
    pretrained=True,
    dropout=0.30
):

    return R18EnhancedFPNClassifier(

        num_classes=NUM_CLASSES,

        pretrained=pretrained,

        dropout=dropout

    )


print("=" * 75)
print("EXPERIMENT 21 - RESNET18 + FPN IMPLEMENTED")
print("=" * 75)

print("Backbone: ResNet18")

print("Pretrained: ImageNet")

print("FPN: Existing generalized FPNHead")

print("FPN channels: 256")

print("Classes:", NUM_CLASSES)

print("\nExisting three architectures remain unchanged.")


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 2
# RESNET18 CONFIGURATION
# ============================================================

import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    log_loss,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# Verify Experiment 20 preparation
# ------------------------------------------------------------

required_globals = [
    "enhanced_df",
    "enhanced_local_path",
    "EnhancedStitchingDataset",
    "FPNHead",
    "create_resnet18_fpn_model",
    "CLASS_WEIGHTS",
    "NUM_CLASSES",
    "class_names",
    "IMAGE_SIZE",
    "DEVICE",
    "SEED",
    "NUM_WORKERS"
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

assert not missing_globals, (
    f"Missing dependencies: {missing_globals}"
)

assert len(enhanced_df) == 14565

assert set(enhanced_df["split"]) == {
    "train", "validation", "test"
}

assert NUM_CLASSES == 11

assert set(enhanced_df["class_id"].astype(int)) == set(
    range(NUM_CLASSES)
)

for variant in [
    "lab_clahe",
    "bilateral_clahe"
]:

    assert f"{variant}_s3_key" in enhanced_df.columns

# ------------------------------------------------------------
# Independent architecture configuration
# ------------------------------------------------------------

R18_ENH_ARCHITECTURE = "resnet18_fpn"

R18_ENH_VARIANTS = [
    "lab_clahe",
    "bilateral_clahe"
]

R18_ENH_CONFIG = {

    "batch_size": 24,
    "accumulation_steps": 1,

    "max_epochs": 60,
    "min_epochs": 10,
    "early_stopping_patience": 10,

    "backbone_lr": 5e-5,
    "head_lr": 3e-4,

    "weight_decay": 1e-4,

    "dropout": 0.30,
    "label_smoothing": 0.03,

    "scheduler_patience": 3,
    "scheduler_factor": 0.5,

    "min_backbone_lr": 1e-6,
    "min_head_lr": 1e-5

}

R18_NUM_WORKERS = min(
    4,
    os.cpu_count() or 1
)

R18_AMP_ENABLED = DEVICE.type == "cuda"

R18_CLASS_NAMES = [
    class_names[i]
    for i in range(NUM_CLASSES)
]

# ------------------------------------------------------------
# Independent output directories
# ------------------------------------------------------------

R18_ENH_MODEL_ROOT = (
    Path.cwd()
    / "models"
    / "experiment21_resnet18_enhanced_fpn"
)

R18_ENH_REPORT_ROOT = (
    Path.cwd()
    / "reports"
    / "model_evaluation"
    / "experiment21_resnet18_enhanced_fpn"
)

R18_ENH_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

R18_ENH_REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Configuration report
# ------------------------------------------------------------

print("=" * 75)
print("EXPERIMENT 21 - RESNET18 + FPN")
print("=" * 75)

print("Architecture:", R18_ENH_ARCHITECTURE)
print("Variants:", R18_ENH_VARIANTS)
print("Device:", DEVICE)
print("AMP:", R18_AMP_ENABLED)
print("Workers:", R18_NUM_WORKERS)

for key, value in R18_ENH_CONFIG.items():

    print(f"{key:35s}: {value}")

print("\nModel directory:", R18_ENH_MODEL_ROOT)

print("Report directory:", R18_ENH_REPORT_ROOT)

print("\nEfficientNet configuration remains untouched.")

print("Experiment 21 configuration ready.")


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 3
# RESNET18 ENHANCED DATASET LOADERS
# ============================================================

# ------------------------------------------------------------
# Verify both enhanced datasets
# ------------------------------------------------------------

for variant in R18_ENH_VARIANTS:

    print("\nVERIFYING:", variant)

    column = f"{variant}_s3_key"

    assert enhanced_df[column].notna().all()

    assert enhanced_df[column].nunique() == 14565

    assert all(
        enhanced_local_path(variant, key).exists()
        for key in enhanced_df[column].head(10)
    )

    print("Dataset verified.")
    print("Images:", len(enhanced_df))

# ------------------------------------------------------------
# Independent loader function
# ------------------------------------------------------------

def r18_make_enhanced_loaders(
    variant,
    batch_size
):

    assert variant in R18_ENH_VARIANTS

    train_metadata = (
        enhanced_df[
            enhanced_df["split"] == "train"
        ]
        .reset_index(drop=True)
    )

    val_metadata = (
        enhanced_df[
            enhanced_df["split"] == "validation"
        ]
        .reset_index(drop=True)
    )

    test_metadata = (
        enhanced_df[
            enhanced_df["split"] == "test"
        ]
        .reset_index(drop=True)
    )

    train_dataset = EnhancedStitchingDataset(
        train_metadata,
        variant,
        training=True
    )

    val_dataset = EnhancedStitchingDataset(
        val_metadata,
        variant,
        training=False
    )

    test_dataset = EnhancedStitchingDataset(
        test_metadata,
        variant,
        training=False
    )

    generator = torch.Generator()

    generator.manual_seed(SEED)

    common = {

        "batch_size": batch_size,

        "num_workers": R18_NUM_WORKERS,

        "pin_memory": DEVICE.type == "cuda",

        "persistent_workers": R18_NUM_WORKERS > 0

    }

    if R18_NUM_WORKERS > 0:

        common["prefetch_factor"] = 2

    train_loader = DataLoader(

        train_dataset,

        shuffle=True,

        generator=generator,

        **common

    )

    val_loader = DataLoader(

        val_dataset,

        shuffle=False,

        **common

    )

    test_loader = DataLoader(

        test_dataset,

        shuffle=False,

        **common

    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


# ------------------------------------------------------------
# Dataset distribution
# ------------------------------------------------------------

print("\nDATASET DISTRIBUTION")

display(
    enhanced_df.groupby(
        ["split", "class_id"]
    ).size().unstack(fill_value=0)
)

print("\nTOTAL IMAGES:", len(enhanced_df))

print("TRAIN:", (enhanced_df["split"] == "train").sum())

print("VALIDATION:", (enhanced_df["split"] == "validation").sum())

print("TEST:", (enhanced_df["split"] == "test").sum())

print("\nResNet18 enhanced DataLoader function ready.")


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 4
# RESNET18 TRAINING ENGINE
# ============================================================

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def r18_seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# Mixed precision
# ------------------------------------------------------------

def r18_autocast():

    if R18_AMP_ENABLED:

        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )

    return nullcontext()


def r18_get_scaler():

    try:

        return torch.amp.GradScaler(
            "cuda",
            enabled=R18_AMP_ENABLED
        )

    except (AttributeError, TypeError):

        return torch.cuda.amp.GradScaler(
            enabled=R18_AMP_ENABLED
        )


# ------------------------------------------------------------
# Training one epoch
# ------------------------------------------------------------

def r18_train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    accumulation_steps
):

    model.train()

    total_loss = 0.0

    total_images = 0

    optimizer.zero_grad(
        set_to_none=True
    )

    for step, (images, labels, _) in enumerate(loader):

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        group_start = (
            step // accumulation_steps
        ) * accumulation_steps

        group_size = min(
            accumulation_steps,
            len(loader) - group_start
        )

        with r18_autocast():

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            scaled_loss = loss / group_size

        scaler.scale(
            scaled_loss
        ).backward()

        should_update = (

            (step + 1) % accumulation_steps == 0

            or step + 1 == len(loader)

        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        total_loss += loss.item() * len(labels)

        total_images += len(labels)

    return total_loss / total_images


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

@torch.inference_mode()
def r18_validate_model(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0

    total_images = 0

    actual = []

    predicted = []

    for images, labels, _ in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        with r18_autocast():

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        predictions = logits.argmax(dim=1)

        total_loss += loss.item() * len(labels)

        total_images += len(labels)

        actual.extend(
            labels.cpu().numpy()
        )

        predicted.extend(
            predictions.cpu().numpy()
        )

    actual = np.asarray(actual)

    predicted = np.asarray(predicted)

    return {

        "val_loss": total_loss / total_images,

        "val_accuracy": accuracy_score(
            actual,
            predicted
        ),

        "val_macro_f1": f1_score(
            actual,
            predicted,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        )

    }


# ============================================================
# COMPLETE RESNET18 TRAINING FUNCTION
# ============================================================

def r18_train_enhanced_variant(variant):

    cfg = R18_ENH_CONFIG.copy()

    r18_seed_everything(SEED)

    train_loader, val_loader, _ = (
        r18_make_enhanced_loaders(
            variant,
            cfg["batch_size"]
        )
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = create_resnet18_fpn_model(

        pretrained=True,

        dropout=cfg["dropout"]

    )

    model = model.to(
        DEVICE,
        memory_format=torch.channels_last
    )

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(

        [

            {
                "params": model.backbone.parameters(),
                "lr": cfg["backbone_lr"]
            },

            {
                "params": model.fpn_head.parameters(),
                "lr": cfg["head_lr"]
            }

        ],

        weight_decay=cfg["weight_decay"]

    )

    # --------------------------------------------------------
    # Weighted loss
    # --------------------------------------------------------

    class_weights = torch.as_tensor(
        CLASS_WEIGHTS,
        dtype=torch.float32,
        device=DEVICE
    )

    criterion = nn.CrossEntropyLoss(

        weight=class_weights,

        label_smoothing=cfg["label_smoothing"]

    )

    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=cfg["scheduler_factor"],

        patience=cfg["scheduler_patience"],

        min_lr=[
            cfg["min_backbone_lr"],
            cfg["min_head_lr"]
        ]

    )

    scaler = r18_get_scaler()

    # --------------------------------------------------------
    # Independent output directory
    # --------------------------------------------------------

    run_dir = (
        R18_ENH_MODEL_ROOT / variant
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    best_path = (
        run_dir / "best_model.pth"
    )

    history_path = (
        run_dir / "training_history.csv"
    )

    history = []

    best_f1 = -1.0

    best_loss = float("inf")

    bad_epochs = 0

    start_time = time.time()

    if DEVICE.type == "cuda":

        torch.cuda.reset_peak_memory_stats(DEVICE)

    print("\n" + "=" * 80)

    print("TRAINING:", variant)

    print("ARCHITECTURE:", R18_ENH_ARCHITECTURE)

    print("=" * 80)

    # --------------------------------------------------------
    # Epoch loop
    # --------------------------------------------------------

    for epoch in range(
        1,
        cfg["max_epochs"] + 1
    ):

        epoch_start = time.time()

        train_loss = r18_train_one_epoch(

            model,

            train_loader,

            optimizer,

            criterion,

            scaler,

            cfg["accumulation_steps"]

        )

        val_metrics = r18_validate_model(

            model,

            val_loader,

            criterion

        )

        current_f1 = val_metrics["val_macro_f1"]

        current_loss = val_metrics["val_loss"]

        scheduler.step(current_f1)

        improved = (

            current_f1 > best_f1 + 1e-5

            or (

                abs(current_f1 - best_f1) <= 1e-5

                and current_loss < best_loss

            )

        )

        # ----------------------------------------------------
        # Save best model
        # ----------------------------------------------------

        if improved:

            best_f1 = current_f1

            best_loss = current_loss

            bad_epochs = 0

            checkpoint = {

                "architecture": R18_ENH_ARCHITECTURE,

                "variant": variant,

                "model_state_dict": {

                    k: v.detach().cpu().clone()

                    for k, v in model.state_dict().items()

                },

                "class_names": R18_CLASS_NAMES,

                "num_classes": NUM_CLASSES,

                "image_size": IMAGE_SIZE,

                "imagenet_mean": list(IMAGENET_MEAN),

                "imagenet_std": list(IMAGENET_STD),

                "best_epoch": epoch,

                "best_val_macro_f1": best_f1,

                "best_val_loss": best_loss,

                "config": cfg

            }

            torch.save(
                checkpoint,
                best_path
            )

        else:

            bad_epochs += 1

        # ----------------------------------------------------
        # Epoch history
        # ----------------------------------------------------

        epoch_seconds = time.time() - epoch_start

        history.append({

            "epoch": epoch,

            "train_loss": train_loss,

            "val_loss": current_loss,

            "val_accuracy": val_metrics["val_accuracy"],

            "val_macro_f1": current_f1,

            "backbone_lr": optimizer.param_groups[0]["lr"],

            "head_lr": optimizer.param_groups[1]["lr"],

            "epoch_seconds": epoch_seconds

        })

        pd.DataFrame(history).to_csv(
            history_path,
            index=False
        )

        print(

            f"Epoch {epoch:02d}/{cfg['max_epochs']} | "

            f"Train Loss {train_loss:.5f} | "

            f"Val Loss {current_loss:.5f} | "

            f"Val Accuracy {val_metrics['val_accuracy']:.5f} | "

            f"Val Macro F1 {current_f1:.5f} | "

            f"Best F1 {best_f1:.5f} | "

            f"Time {epoch_seconds:.1f}s"

        )

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if (

            epoch >= cfg["min_epochs"]

            and bad_epochs >= cfg["early_stopping_patience"]

        ):

            print("\nEarly stopping activated.")

            break

    # --------------------------------------------------------
    # Training summary
    # --------------------------------------------------------

    training_seconds = time.time() - start_time

    peak_memory_gb = (

        torch.cuda.max_memory_allocated(DEVICE) / 1024**3

        if DEVICE.type == "cuda"

        else 0.0

    )

    best_epoch = max(
        history,
        key=lambda x: (
            x["val_macro_f1"],
            -x["val_loss"]
        )
    )["epoch"]

    result = {

        "architecture": R18_ENH_ARCHITECTURE,

        "variant": variant,

        "best_val_macro_f1": best_f1,

        "best_val_loss": best_loss,

        "best_epoch": int(best_epoch),

        "training_minutes": training_seconds / 60,

        "peak_gpu_memory_gb": peak_memory_gb,

        "checkpoint": str(best_path)

    }

    with open(
        run_dir / "config.json",
        "w"
    ) as f:

        json.dump(
            {
                **cfg,
                **result
            },
            f,
            indent=4
        )

    del model

    del train_loader

    del val_loader

    gc.collect()

    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()

    print("\nTRAINING COMPLETED:", variant)

    print("Best validation F1:", best_f1)

    print("Best epoch:", best_epoch)

    print("Training minutes:", round(training_seconds / 60, 2))

    print("Checkpoint:", best_path)

    return result


print("\nExperiment 21 training engine ready.")


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 5
# RUN BOTH RESNET18 ENHANCED EXPERIMENTS
# ============================================================

R18_ENH_TRAINING_RESULTS = []

for variant in R18_ENH_VARIANTS:

    print("\n" + "#" * 85)

    print("STARTING EXPERIMENT:", variant)

    print("#" * 85)

    result = r18_train_enhanced_variant(
        variant
    )

    R18_ENH_TRAINING_RESULTS.append(result)

    pd.DataFrame(
        R18_ENH_TRAINING_RESULTS
    ).to_csv(

        R18_ENH_MODEL_ROOT /
        "resnet18_enhanced_training_summary.csv",

        index=False

    )

print("\n" + "=" * 85)

print("EXPERIMENT 21 TRAINING COMPLETED")

print("=" * 85)

R18_ENH_TRAINING_SUMMARY = pd.DataFrame(
    R18_ENH_TRAINING_RESULTS
)

display(R18_ENH_TRAINING_SUMMARY)


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 6
# COMPLETE RESNET18 EVALUATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score
)

# ------------------------------------------------------------
# Load trained ResNet18
# ------------------------------------------------------------

def r18_load_enhanced_model(variant):

    checkpoint_path = (
        R18_ENH_MODEL_ROOT
        / variant
        / "best_model.pth"
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint: {checkpoint_path}"
    )

    checkpoint = torch.load(

        checkpoint_path,

        map_location="cpu",

        weights_only=True

    )

    model = create_resnet18_fpn_model(

        pretrained=False,

        dropout=checkpoint["config"]["dropout"]

    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(
        DEVICE,
        memory_format=torch.channels_last
    )

    model.eval()

    return model, checkpoint


# ------------------------------------------------------------
# Evaluation DataLoader
# ------------------------------------------------------------

def r18_make_eval_loader(
    variant,
    metadata
):

    dataset = EnhancedStitchingDataset(

        metadata.reset_index(drop=True),

        variant,

        training=False

    )

    loader_kwargs = {

        "batch_size": R18_ENH_CONFIG["batch_size"],

        "shuffle": False,

        "num_workers": R18_NUM_WORKERS,

        "pin_memory": DEVICE.type == "cuda",

        "persistent_workers": R18_NUM_WORKERS > 0

    }

    if R18_NUM_WORKERS > 0:

        loader_kwargs["prefetch_factor"] = 2

    return DataLoader(
        dataset,
        **loader_kwargs
    )


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def r18_calculate_ece(
    probabilities,
    targets,
    bins=15
):

    confidence = probabilities.max(axis=1)

    predictions = probabilities.argmax(axis=1)

    correctness = (
        predictions == targets
    )

    ece = 0.0

    boundaries = np.linspace(
        0,
        1,
        bins + 1
    )

    for i in range(bins):

        mask = (

            (confidence > boundaries[i])

            & (confidence <= boundaries[i + 1])

        )

        if mask.sum() == 0:

            continue

        accuracy = correctness[mask].mean()

        avg_confidence = confidence[mask].mean()

        ece += (

            mask.mean()

            * abs(accuracy - avg_confidence)

        )

    return float(ece)


# ------------------------------------------------------------
# Collect predictions and inference timing
# ------------------------------------------------------------

@torch.inference_mode()
def r18_collect_predictions(
    model,
    loader
):

    model.eval()

    y_true = []

    y_pred = []

    y_prob = []

    indices = []

    batch_latencies = []

    inference_seconds = 0.0

    if DEVICE.type == "cuda":

        torch.cuda.reset_peak_memory_stats(DEVICE)

    for images, labels, batch_indices in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        if DEVICE.type == "cuda":

            torch.cuda.synchronize(DEVICE)

        start = time.perf_counter()

        with r18_autocast():

            logits = model(images)

        if DEVICE.type == "cuda":

            torch.cuda.synchronize(DEVICE)

        elapsed = time.perf_counter() - start

        inference_seconds += elapsed

        batch_latencies.extend(
            [elapsed * 1000 / len(images)] * len(images)
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )

        predictions = probabilities.argmax(
            dim=1
        )

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            predictions.cpu().numpy()
        )

        y_prob.extend(
            probabilities.cpu().numpy()
        )

        indices.extend(
            batch_indices.numpy()
        )

    peak_gpu_memory_mb = (

        torch.cuda.max_memory_allocated(DEVICE) / 1024**2

        if DEVICE.type == "cuda"

        else 0.0

    )

    return {

        "y_true": np.asarray(y_true),

        "y_pred": np.asarray(y_pred),

        "y_prob": np.asarray(y_prob),

        "indices": np.asarray(indices),

        "inference_seconds": inference_seconds,

        "latencies": np.asarray(batch_latencies),

        "peak_gpu_memory_mb": peak_gpu_memory_mb

    }


# ------------------------------------------------------------
# Calculate complete metrics
# ------------------------------------------------------------

def r18_evaluate_predictions(
    prediction,
    metadata,
    variant,
    split_name,
    checkpoint_path,
    training_minutes,
    model
):

    y_true = prediction["y_true"]

    y_pred = prediction["y_pred"]

    y_prob = prediction["y_prob"]

    indices = prediction["indices"]

    n = len(y_true)

    labels = np.arange(NUM_CLASSES)

    one_hot = np.eye(NUM_CLASSES)[y_true]

    # --------------------------------------------------------
    # Main metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    kappa = cohen_kappa_score(
        y_true,
        y_pred
    )

    # --------------------------------------------------------
    # Probability metrics
    # --------------------------------------------------------

    nll = log_loss(
        y_true,
        y_prob,
        labels=labels
    )

    brier = np.mean(
        np.sum(
            (y_prob - one_hot) ** 2,
            axis=1
        )
    )

    ece = r18_calculate_ece(
        y_prob,
        y_true
    )

    top2 = np.argsort(
        y_prob,
        axis=1
    )[:, -2:]

    top2_accuracy = np.mean(
        [
            y_true[i] in top2[i]
            for i in range(n)
        ]
    )

    try:

        macro_roc_auc = roc_auc_score(

            y_true,

            y_prob,

            labels=labels,

            multi_class="ovr",

            average="macro"

        )

    except ValueError:

        macro_roc_auc = np.nan

    try:

        macro_pr_auc = average_precision_score(
            one_hot,
            y_prob,
            average="macro"
        )

    except ValueError:

        macro_pr_auc = np.nan

    # --------------------------------------------------------
    # Inference metrics
    # --------------------------------------------------------

    inference_seconds = prediction["inference_seconds"]

    latencies = prediction["latencies"]

    avg_latency = (
        inference_seconds * 1000 / n
    )

    p50_latency = np.percentile(
        latencies,
        50
    )

    p95_latency = np.percentile(
        latencies,
        95
    )

    throughput = (
        n / inference_seconds
        if inference_seconds > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Model size and parameters
    # --------------------------------------------------------

    parameters = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_parameters = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    model_size_mb = (
        Path(checkpoint_path).stat().st_size
        / 1024**2
    )

    # --------------------------------------------------------
    # Fabric-wise performance
    # --------------------------------------------------------

    evaluated_metadata = (
        metadata.iloc[indices]
        .reset_index(drop=True)
    )

    fabric_column = next(
        (
            column
            for column in [
                "fabric",
                "fabric_name",
                "fabric_type"
            ]
            if column in evaluated_metadata.columns
        ),
        None
    )

    fabric_records = []

    if fabric_column is not None:

        fabric_values = evaluated_metadata[
            fabric_column
        ].astype(str).to_numpy()

        for fabric in sorted(
            set(fabric_values)
        ):

            mask = fabric_values == fabric

            fabric_records.append({

                "fabric": fabric,

                "accuracy": accuracy_score(
                    y_true[mask],
                    y_pred[mask]
                ),

                "macro_f1": f1_score(
                    y_true[mask],
                    y_pred[mask],
                    labels=labels,
                    average="macro",
                    zero_division=0
                ),

                "support": int(mask.sum()),

                "variant": variant,

                "split": split_name

            })

    per_fabric = pd.DataFrame(
        fabric_records
    )

    worst_fabric_f1 = (

        per_fabric["macro_f1"].min()

        if not per_fabric.empty

        else np.nan

    )

    # --------------------------------------------------------
    # Per-defect metrics
    # --------------------------------------------------------

    precision, recall, f1, support = (
        precision_recall_fscore_support(

            y_true,

            y_pred,

            labels=labels,

            zero_division=0

        )
    )

    per_class = pd.DataFrame({

        "class_id": labels,

        "class_name": R18_CLASS_NAMES,

        "precision": precision,

        "recall": recall,

        "f1_score": f1,

        "support": support,

        "variant": variant,

        "split": split_name

    })

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    # --------------------------------------------------------
    # Overall metrics
    # --------------------------------------------------------

    metrics = {

        "model": R18_ENH_ARCHITECTURE,

        "architecture": R18_ENH_ARCHITECTURE,

        "variant": variant,

        "split": split_name,

        "accuracy": accuracy,

        "macro_precision": macro_precision,

        "macro_recall": macro_recall,

        "macro_f1": macro_f1,

        "weighted_f1": weighted_f1,

        "balanced_accuracy": balanced_accuracy,

        "mcc": mcc,

        "cohen_kappa": kappa,

        "negative_log_likelihood": nll,

        "multiclass_brier_score": brier,

        "ece": ece,

        "top2_accuracy": top2_accuracy,

        "macro_roc_auc": macro_roc_auc,

        "macro_pr_auc": macro_pr_auc,

        "inference_seconds": inference_seconds,

        "avg_latency_ms_per_image": avg_latency,

        "p50_latency_ms_per_image": p50_latency,

        "p95_latency_ms_per_image": p95_latency,

        "throughput_images_per_second": throughput,

        "parameters": parameters,

        "trainable_parameters": trainable_parameters,

        "model_size_mb": model_size_mb,

        "peak_gpu_memory_mb": prediction["peak_gpu_memory_mb"],

        "worst_fabric_macro_f1": worst_fabric_f1,

        "training_minutes": training_minutes,

        "evaluated_images": n

    }

    # --------------------------------------------------------
    # Individual predictions
    # --------------------------------------------------------

    predictions_df = evaluated_metadata.copy()

    predictions_df["actual_class_id"] = y_true

    predictions_df["predicted_class_id"] = y_pred

    predictions_df["actual_class"] = [
        R18_CLASS_NAMES[i] for i in y_true
    ]

    predictions_df["predicted_class"] = [
        R18_CLASS_NAMES[i] for i in y_pred
    ]

    predictions_df["confidence"] = y_prob.max(axis=1)

    predictions_df["correct"] = y_true == y_pred

    predictions_df["top2_classes"] = [
        " | ".join(
            R18_CLASS_NAMES[j]
            for j in row[::-1]
        )
        for row in top2
    ]

    for class_id in labels:

        predictions_df[
            f"probability_class_{class_id}"
        ] = y_prob[:, class_id]

    return {

        "metrics": metrics,

        "per_class": per_class,

        "per_fabric": per_fabric,

        "confusion_matrix": cm,

        "predictions": predictions_df

    }


# ============================================================
# EVALUATE BOTH VARIANTS
# ============================================================

R18_ENH_METRICS_RECORDS = []

R18_ENH_CLASS_RECORDS = []

R18_ENH_FABRIC_RECORDS = []

R18_ENH_EVALUATION_RESULTS = {}

for variant in R18_ENH_VARIANTS:

    print("\n" + "=" * 85)

    print("EVALUATING:", variant)

    print("=" * 85)

    model, checkpoint = r18_load_enhanced_model(
        variant
    )

    training_minutes = next(

        r["training_minutes"]

        for r in R18_ENH_TRAINING_RESULTS

        if r["variant"] == variant

    )

    variant_results = {}

    for split_name in [
        "validation",
        "test"
    ]:

        metadata = (
            enhanced_df[
                enhanced_df["split"] == split_name
            ]
            .reset_index(drop=True)
        )

        loader = r18_make_eval_loader(
            variant,
            metadata
        )

        prediction = r18_collect_predictions(
            model,
            loader
        )

        result = r18_evaluate_predictions(

            prediction,

            metadata,

            variant,

            split_name,

            R18_ENH_MODEL_ROOT
            / variant
            / "best_model.pth",

            training_minutes,

            model

        )

        variant_results[split_name] = result

        R18_ENH_METRICS_RECORDS.append(
            result["metrics"]
        )

        R18_ENH_CLASS_RECORDS.append(
            result["per_class"]
        )

        R18_ENH_FABRIC_RECORDS.append(
            result["per_fabric"]
        )

        # ----------------------------------------------------
        # Save confusion matrix
        # ----------------------------------------------------

        cm_df = pd.DataFrame(

            result["confusion_matrix"],

            index=R18_CLASS_NAMES,

            columns=R18_CLASS_NAMES

        )

        cm_df.to_csv(

            R18_ENH_REPORT_ROOT
            / f"{variant}_{split_name}_confusion_matrix.csv"

        )

        # ----------------------------------------------------
        # Save predictions
        # ----------------------------------------------------

        result["predictions"].to_csv(

            R18_ENH_REPORT_ROOT
            / f"{variant}_{split_name}_predictions.csv",

            index=False

        )

        # ----------------------------------------------------
        # Save per-class metrics
        # ----------------------------------------------------

        result["per_class"].to_csv(

            R18_ENH_REPORT_ROOT
            / f"{variant}_{split_name}_per_class.csv",

            index=False

        )

        # ----------------------------------------------------
        # Save per-fabric metrics
        # ----------------------------------------------------

        result["per_fabric"].to_csv(

            R18_ENH_REPORT_ROOT
            / f"{variant}_{split_name}_per_fabric.csv",

            index=False

        )

        print(

            split_name,

            "| Accuracy:",

            round(result["metrics"]["accuracy"], 6),

            "| Macro F1:",

            round(result["metrics"]["macro_f1"], 6),

            "| ECE:",

            round(result["metrics"]["ece"], 6),

            "| Worst Fabric F1:",

            round(
                result["metrics"]["worst_fabric_macro_f1"],
                6
            )

        )

        del loader

    R18_ENH_EVALUATION_RESULTS[variant] = variant_results

    del model

    gc.collect()

    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()


# ============================================================
# CONSOLIDATED REPORTS
# ============================================================

R18_ENH_METRICS_DF = pd.DataFrame(
    R18_ENH_METRICS_RECORDS
)

R18_ENH_CLASS_DF = pd.concat(
    R18_ENH_CLASS_RECORDS,
    ignore_index=True
)

R18_ENH_FABRIC_DF = pd.concat(
    R18_ENH_FABRIC_RECORDS,
    ignore_index=True
)

R18_ENH_METRICS_DF.to_csv(

    R18_ENH_REPORT_ROOT /
    "resnet18_enhanced_all_metrics.csv",

    index=False

)

R18_ENH_CLASS_DF.to_csv(

    R18_ENH_REPORT_ROOT /
    "resnet18_enhanced_per_class_metrics.csv",

    index=False

)

R18_ENH_FABRIC_DF.to_csv(

    R18_ENH_REPORT_ROOT /
    "resnet18_enhanced_fabric_metrics.csv",

    index=False

)

print("\n" + "=" * 85)

print("RESNET18 COMPLETE EVALUATION FINISHED")

print("=" * 85)

display(

    R18_ENH_METRICS_DF[

        [

            "model",

            "variant",

            "split",

            "accuracy",

            "macro_precision",

            "macro_recall",

            "macro_f1",

            "weighted_f1",

            "balanced_accuracy",

            "mcc",

            "ece",

            "top2_accuracy",

            "macro_roc_auc",

            "macro_pr_auc",

            "worst_fabric_macro_f1",

            "throughput_images_per_second"

        ]

    ]

)


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 7
# ENHANCED INPUT COMPARISON
# ============================================================

assert len(R18_ENH_METRICS_DF) == 4

comparison_columns = [

    "variant",

    "split",

    "accuracy",

    "macro_precision",

    "macro_recall",

    "macro_f1",

    "weighted_f1",

    "balanced_accuracy",

    "mcc",

    "ece",

    "macro_roc_auc",

    "macro_pr_auc",

    "worst_fabric_macro_f1",

    "throughput_images_per_second"

]

print("=" * 90)

print("RESNET18 + FPN - ENHANCED INPUT COMPARISON")

print("=" * 90)

display(

    R18_ENH_METRICS_DF[
        comparison_columns
    ].sort_values(
        ["split", "macro_f1"],
        ascending=[True, False]
    )

)

# ------------------------------------------------------------
# Validation-only ranking
# ------------------------------------------------------------

validation_ranking = (

    R18_ENH_METRICS_DF[

        R18_ENH_METRICS_DF["split"] == "validation"

    ]

    .sort_values(

        [

            "macro_f1",

            "worst_fabric_macro_f1",

            "balanced_accuracy",

            "ece"

        ],

        ascending=[

            False,

            False,

            False,

            True

        ],

        na_position="last"

    )

    .reset_index(drop=True)

)

validation_ranking.to_csv(

    R18_ENH_REPORT_ROOT /
    "resnet18_validation_input_ranking.csv",

    index=False

)

print("\nVALIDATION-BASED RANKING")

display(
    validation_ranking[comparison_columns]
)

# ------------------------------------------------------------
# Select winning input representation
# ------------------------------------------------------------

R18_ENH_WINNING_VARIANT = (
    validation_ranking.iloc[0]["variant"]
)

print("\n" + "=" * 80)

print("WINNING RESNET18 INPUT REPRESENTATION")

print("=" * 80)

print("Winner:", R18_ENH_WINNING_VARIANT)

print(
    "Validation Macro F1:",
    round(
        validation_ranking.iloc[0]["macro_f1"],
        6
    )
)

print(
    "Validation Accuracy:",
    round(
        validation_ranking.iloc[0]["accuracy"],
        6
    )
)

print(
    "Validation ECE:",
    round(
        validation_ranking.iloc[0]["ece"],
        6
    )
)

# ------------------------------------------------------------
# Final test results for both variants
# ------------------------------------------------------------

test_comparison = (

    R18_ENH_METRICS_DF[

        R18_ENH_METRICS_DF["split"] == "test"

    ]

    .sort_values(
        "variant"
    )

    .reset_index(drop=True)

)

print("\nFINAL TEST PERFORMANCE")

display(
    test_comparison[comparison_columns]
)

# ------------------------------------------------------------
# Defect-wise comparison
# ------------------------------------------------------------

print("\nDEFECT-WISE F1 COMPARISON")

defect_pivot = R18_ENH_CLASS_DF.pivot(

    index="class_name",

    columns=["variant", "split"],

    values="f1_score"

)

display(
    defect_pivot.style.format("{:.5f}")
)

defect_pivot.to_csv(

    R18_ENH_REPORT_ROOT /
    "resnet18_defect_comparison.csv"

)

# ------------------------------------------------------------
# Fabric-wise comparison
# ------------------------------------------------------------

if not R18_ENH_FABRIC_DF.empty:

    print("\nFABRIC-WISE MACRO F1 COMPARISON")

    fabric_pivot = R18_ENH_FABRIC_DF.pivot(

        index="fabric",

        columns=["variant", "split"],

        values="macro_f1"

    )

    display(
        fabric_pivot.style.format("{:.5f}")
    )

    fabric_pivot.to_csv(

        R18_ENH_REPORT_ROOT /
        "resnet18_fabric_comparison.csv"

    )

# ------------------------------------------------------------
# Save consolidated comparison
# ------------------------------------------------------------

R18_ENH_METRICS_DF.to_csv(

    R18_ENH_REPORT_ROOT /
    "resnet18_original_enhanced_input_metrics.csv",

    index=False

)

print("\nEXPERIMENT 21 COMPARISON COMPLETED.")


In [ ]:

# ============================================================
# EXPERIMENT 21 - CELL 8
# RESNET18 ENHANCED MODELS S3 BACKUP
# ============================================================

assert "s3" in globals()

assert "BUCKET_NAME" in globals()

R18_S3_EXPERIMENT_PREFIX = (

    "model-artifacts/"

    "stitching-defect-detection/"

    "experiment21_resnet18_enhanced_fpn/"

)

# ------------------------------------------------------------
# Collect model and report files
# ------------------------------------------------------------

r18_upload_files = [

    p

    for p in R18_ENH_MODEL_ROOT.rglob("*")

    if p.is_file()

]

r18_upload_files += [

    p

    for p in R18_ENH_REPORT_ROOT.rglob("*")

    if p.is_file()

]

r18_upload_files = sorted(
    set(r18_upload_files)
)

assert len(r18_upload_files) > 0

# ------------------------------------------------------------
# Upload files
# ------------------------------------------------------------

r18_upload_manifest = []

for path in tqdm(

    r18_upload_files,

    desc="Uploading Experiment 21"

):

    if path.is_relative_to(
        R18_ENH_MODEL_ROOT
    ):

        relative = path.relative_to(
            R18_ENH_MODEL_ROOT
        )

        category = "models"

    else:

        relative = path.relative_to(
            R18_ENH_REPORT_ROOT
        )

        category = "reports"

    s3_key = (

        R18_S3_EXPERIMENT_PREFIX

        + category

        + "/"

        + relative.as_posix()

    )

    s3.upload_file(

        str(path),

        BUCKET_NAME,

        s3_key

    )

    r18_upload_manifest.append({

        "local_path": str(path),

        "s3_key": s3_key,

        "size_mb": (
            path.stat().st_size / 1024**2
        )

    })

# ------------------------------------------------------------
# Save upload manifest
# ------------------------------------------------------------

R18_UPLOAD_MANIFEST_DF = pd.DataFrame(
    r18_upload_manifest
)

manifest_path = (

    R18_ENH_REPORT_ROOT /

    "s3_upload_manifest.csv"

)

R18_UPLOAD_MANIFEST_DF.to_csv(

    manifest_path,

    index=False

)

# Upload manifest separately

s3.upload_file(

    str(manifest_path),

    BUCKET_NAME,

    R18_S3_EXPERIMENT_PREFIX
    + "reports/s3_upload_manifest.csv"

)

# ------------------------------------------------------------
# Backup summary
# ------------------------------------------------------------

print("\n" + "=" * 85)

print("EXPERIMENT 21 S3 BACKUP COMPLETED")

print("=" * 85)

print("Uploaded files:", len(R18_UPLOAD_MANIFEST_DF))

print("Bucket:", BUCKET_NAME)

print("S3 prefix:", R18_S3_EXPERIMENT_PREFIX)

display(
    R18_UPLOAD_MANIFEST_DF.head(20)
)
